# DES Y6: exploring the 3×2pt data vector

This notebook computes the lens-galaxy predictions of the DES Y6 project with
`cosmolike_des_y6_interface`, the project's *compiled interface*: C and C++ code
built into a Python module (through the pybind11 library), imported below as
`ci`. It shows

- galaxy–galaxy lensing, the correlation between lens-galaxy positions and
  source-galaxy shear, as the angular spectrum $C_\ell^{gs}$ and the tangential
  shear $\gamma_t(\theta)$;
- galaxy clustering, the correlation of lens-galaxy positions, as $C_\ell^{gg}$,
  with and without the Limber approximation, and as the angular correlation
  function $w(\theta)$;
- how $\Omega_m$, $n_s$, $A_s$, the angular binning, baryonic feedback and the
  numerical accuracy settings change them;
- the $\chi^2$ of a prediction in units of the forecast DES Y6 errors, against a
  synthetic data vector, for the accuracy settings and for the
  baryonic-feedback models of the `bfmt` theory block.

"3×2pt" names the three two-point functions analyzed together: cosmic shear
$\xi_\pm$ (explored in EXAMPLE_EVALUATE2.ipynb), galaxy–galaxy lensing and galaxy
clustering.

**Vocabulary.**

- A *tomographic bin* is one redshift slice of a galaxy sample: six *lens* bins,
  galaxies used as density tracers, and four *source* bins, galaxies whose
  shapes measure the gravitational shear. Redshifts come from broad-band colors
  (*photo-z*).
- The multipole $\ell$ is an angular frequency, roughly $180^\circ/\theta$;
  $C_\ell$ is the angular power spectrum at that frequency.
- The *data vector* is the ordered list of all two-point values: 1,300 entries,
  26 logarithmic angular bins from 2.5 to 995.27 arcmin for $\xi_+$ and $\xi_-$
  of the 10 source-bin pairs (260 entries each), $\gamma_t$ of all 24 (lens,
  source) pairs (624) and $w$ of the six lens bins (156).
- A *mask* holds one 0/1 flag per data-vector entry. The DES Y6 scale cuts,
  `data/DESY6.mask`, keep 541 entries: 152, 58, 272 and 59 of the four blocks.
- A *wrapper* is a Python function of this notebook that runs CAMB, hands the
  result and the nuisance parameters to `ci`, and returns NumPy arrays.

**The shipped data are placeholders.** `data/DESY6.dataset` names an all-ones
data vector, an identity covariance and `ones.mask`, which keeps all 1,300
entries, so a $\chi^2$ against them measures nothing. The $\chi^2$ sections
install three other inputs in their place: the project's forecast covariance,
the DES Y6 scale cuts and a synthetic data vector, a prediction of this
notebook.

**Conventions.**

- Angles $\theta$ are in arcminutes throughout.
- `ci` returns plain, dimensionless $C_\ell$, without the factor
  $\ell(\ell+1)/2\pi$, and the spectrum plotters of this notebook draw $C_\ell$
  itself; the real-space plotters draw $|\gamma_t|$ and $|w|$.
- Arrays count bins from 0, panel labels count from 1: the galaxy–galaxy lensing
  panel labeled $(1,2)$, read (lens, source), draws the array slice `[:, 0, 1]`.

**Running.** Follow the [notebook steps of the project README](README.md#notebooks)
(activate Cocoa, set `OMP_NUM_THREADS`, start Jupyter) and run the cells in
order. Every wrapper call runs CAMB once, so a sweep of ten values costs ten
CAMB runs. Two parts need more than the compiled project:

- the $\chi^2$ sections read `covariance/forecast_real.npz`, which
  EXAMPLE_EVALUATE_COVARIANCE.ipynb or `covariance/compute_covariance.py` writes
  with the optional covariance build
  ([computing covariances](README.md#computing_covariances));
- the last section needs the `bfmt` theory block and its emulators
  ([baryonic feedback](README.md#baryons)).

## Setup: packages, threads and figure style

The first cell imports the packages and sets the figure style; the second adds
CAMB and the shared package `cosmolike_notebook_utils` (imported as `cnu`) to
`sys.path`, the list of folders Python searches for modules, and prints which
copies were found.

- `OMP_NUM_THREADS` sets how many OpenMP threads (parallel workers inside the
  compiled code) CosmoLike uses. An OpenMP runtime reads it once, when it
  starts, so the README sets it in the shell before Jupyter starts; the cell
  sets it again before `ci` is imported. A library loaded in the same process
  can lower the count later, so every wrapper below passes `threads` to
  `ci.set_omp_threads` first, as the likelihood does before each evaluation.
- `text.usetex = True` typesets all figure text with LaTeX and needs a LaTeX
  installation; set it to `False` if the figures fail to render.
- `IniFile` (from GetDist) reads `key = value` text files such as the project's
  `.dataset` file.

In [ ]:
# Figures appear below each cell; 'retina' draws them at double resolution.
%matplotlib inline
%config InlineBackend.figure_format = 'retina'
import sys, platform, os
# OpenMP threads (parallel workers) of the compiled CosmoLike loops. The
# OpenMP runtime reads this variable once, when it starts, so it is set
# before ci is imported; the README sets it in the shell before Jupyter.
# Lower the value on a machine with fewer than eight cores.
os.environ['OMP_NUM_THREADS'] = '8'
import matplotlib
import math
from matplotlib import pyplot as plt
import numpy as np
# EuclidEmulator2: used only when non_linear_emul = 1
import euclidemu2
import scipy
# ci: the project's compiled CosmoLike module
import cosmolike_des_y6_interface as ci
# IniFile reads key = value text files such as the .dataset file
from getdist import IniFile
import itertools
import iminuit
import functools
print(sys.version)
print(os.getcwd())

# Every wrapper hands this count to ci.set_omp_threads before computing: a
# library loaded in the same process can lower the OpenMP team size
threads = int(os.environ['OMP_NUM_THREADS'])

# GENERAL PLOT OPTIONS
matplotlib.rcParams['mathtext.fontset'] = 'stix'
matplotlib.rcParams['font.family'] = 'STIXGeneral'
matplotlib.rcParams['mathtext.rm'] = 'Bitstream Vera Sans'
matplotlib.rcParams['mathtext.it'] = 'Bitstream Vera Sans:italic'
matplotlib.rcParams['mathtext.bf'] = 'Bitstream Vera Sans:bold'
matplotlib.rcParams['xtick.bottom'] = True
matplotlib.rcParams['xtick.top'] = False
matplotlib.rcParams['ytick.right'] = False
matplotlib.rcParams['axes.edgecolor'] = 'black'
matplotlib.rcParams['axes.linewidth'] = '1.0'
matplotlib.rcParams['axes.labelsize'] = 'medium'
matplotlib.rcParams['axes.grid'] = True
matplotlib.rcParams['grid.linewidth'] = '0.0'
matplotlib.rcParams['grid.alpha'] = '0.18'
matplotlib.rcParams['grid.color'] = 'lightgray'
matplotlib.rcParams['legend.labelspacing'] = 0.77
matplotlib.rcParams['savefig.bbox'] = 'tight'
matplotlib.rcParams['savefig.format'] = 'pdf'
# usetex typesets all figure text with LaTeX: it needs a LaTeX installation
matplotlib.rcParams['text.usetex'] = True

# Jupyter Notebook Display options
import IPython
IPython.display.display(IPython.display.HTML("<style>:root { --jp-notebook-max-width: 85% !important; }</style>"))
IPython.display.display(IPython.display.HTML("<style>div.output_scroll { height: 54em; }</style>"))

In [ ]:
# IMPORT CAMB
# This build folder exists only on Linux x86_64. Python skips a missing
# folder in sys.path, so camb may come from another entry; the print shows
# which copy was imported.
sys.path.insert(0, os.environ['ROOTDIR']+'/external_modules/code/CAMB/build/lib.linux-x86_64-'+os.environ['PYTHON_VERSION'])
import camb
from camb import model
print('Using CAMB %s installed at %s'%(camb.__version__,os.path.dirname(camb.__file__)))

# IMPORT SHARED NOTEBOOK UTILITIES (cosmolike_core)
# cnu: the CAMB helper and plotting functions shared by every project
sys.path.insert(0, os.environ['ROOTDIR']+'/external_modules/code/cosmolike_core')
import cosmolike_notebook_utils as cnu
print('Using cosmolike_notebook_utils installed at %s'%(os.path.dirname(cnu.__file__)))

## Run settings

These module-level variables (names defined at the top level of the notebook)
mirror keys of the likelihood YAML files, `likelihood/combo_3x2pt.yaml` and
`EXAMPLE_EVALUATE1.yaml`:

| Variable | Meaning |
| --- | --- |
| `non_linear_emul` | Nonlinear matter power: 1 = EuclidEmulator2, 2 = the CAMB model named by `halofit_version`. |
| `halofit_version` | CAMB's nonlinear model: `"mead2020_feedback"`, HMcode 2020 with its baryonic-feedback term, as in the YAML; `"takahashi"`, the gravity-only Halofit fit of Takahashi et al. 2012. |
| `kmax`, `k_per_logint`, `CAMBAccuracyBoost` | CAMB's largest wavenumber in 1/Mpc, its samples per logarithmic $k$ interval and its accuracy setting, from the YAML's `theory: camb` block. |
| `CLprobe` | Probe combination of the $\chi^2$ sections; `"3x2pt"` is shear, galaxy–galaxy lensing and clustering. |
| `path`, `data_file` | The DES Y6 dataset file; `path` is relative to this notebook's folder. |
| `IA_model` | Intrinsic-alignment model: 0 = NLA, 1 = TATT. |
| `IA_redshift_evolution` | 3 = each IA amplitude follows a power law in $(1+z)/1.62$. |
| `IA_code` | 0 = the C implementation of FAST-PT, the algorithm for the perturbation-theory integrals of TATT. |
| `bias_model` | Redshift model of the six bias terms $(b_1, b_2, b_{s2}, b_3, b_{\rm mag}, b_K)$; code 0 is one constant amplitude per lens bin. |
| `adopt_limber_gs`, `adopt_limber_gg` | 1 = Limber approximation at every multipole, 0 = exact projection below $\ell=150$: Limber for galaxy–galaxy lensing, exact for clustering. |
| `nonlimber_accuracyboost` | Refinement of the comoving-distance grid of the exact projection. |
| `photoz_interpolation_type`, `photoz_zmid_convention` | How the $n(z)$ tables are interpolated, and whether their redshift column holds left bin edges (the DES Y6 files) or bin centers. |
| `internal_accuracyboost` | FAST-PT internal grid, as a fraction of its output table. |
| `integration_accuracy`, `lmax` | Quadrature level of the line-of-sight integrals, and the largest multipole of the $C_\ell$ tables behind the real-space transforms. |
| `ntheta`, `theta_min_arcmin`, `theta_max_arcmin` | 26 logarithmic angular bins from 2.5 to 995.27 arcmin, as in the dataset. |

*Intrinsic alignment* (IA) is the correlation of galaxy shapes with the local
tidal field; it adds to the lensing signal of the sources and must be modeled.
NLA, the nonlinear linear-alignment model of the DES Y6 YAML files, has one
amplitude $A_1$; TATT (tidal alignment and tidal torquing) adds a quadratic
amplitude $A_2$ and $b_{TA}$, which weights the alignment by the source-galaxy
density.

**HMcode feedback.** `EXAMPLE_EVALUATE1.yaml` also sets `HMCode_logT_AGN: 7.7`,
the heating temperature of the AGN feedback in HMcode 2020.
`cnu.get_camb_cosmology` passes only the model name to CAMB, so this notebook's
HMcode runs at CAMB's default, $\log_{10}(T_{\rm AGN}/{\rm K})=7.8$. The two
baryonic-feedback sections pass `halofit_version = "takahashi"`: a feedback
ratio from a simulation or an emulator multiplies a gravity-only spectrum, and
on top of HMcode's feedback term it would count feedback twice (see
[baryonic feedback](README.md#baryons) in the README).

In [ ]:
# Nonlinear power: 1 = EuclidEmulator2, 2 = the CAMB model named below
non_linear_emul = 2
# HMcode 2020 with its feedback term, as in EXAMPLE_EVALUATE1.yaml. The
# YAML's HMCode_logT_AGN = 7.7 does not reach CAMB through
# cnu.get_camb_cosmology, which leaves CAMB's default log10(T_AGN/K) = 7.8.
# The baryonic-feedback sections pass "takahashi", the gravity-only Halofit.
halofit_version = "mead2020_feedback"
# CAMB inputs of the YAML's theory block: kmax in 1/Mpc, samples per
# logarithmic k interval and CAMB's AccuracyBoost. cnu.get_camb_cosmology
# raises kmax and k_per_logint further with the CAMB accuracy.
kmax = 5.0
k_per_logint = 10
CAMBAccuracyBoost = 1.05
# Probe combination of the chi^2 sections: shear, lensing and clustering
CLprobe = "3x2pt"

# Relative to this notebook's folder; start_cocoa.sh links
# external_modules/data/des_y6 to this project's data folder.
path = "../../external_modules/data/des_y6"
data_file = "DESY6.dataset"

# IA_model: 0 = NLA, 1 = TATT. IA_redshift_evolution = 3: power law in
# (1+z)/1.62, so A1 = [amplitude, exponent, ...] and likewise A2.
IA_model = 0
IA_redshift_evolution = 3
IA_code = 0  # 0 = C FASTPT, 1 = Python FAST-PT (NLA always uses 0)

# Redshift model of (b1, b2, bs2, b3, bmag, bK): code 0 = one constant
# amplitude per lens bin (cosmolike bias.c)
bias_model = [0, 0, 0, 0, 0, 0]

# Projection: 1 = Limber at every multipole; 0 = exact below ell = 150
adopt_limber_gs = 1
adopt_limber_gg = 0
# Refinement of the comoving-distance grid of the exact projection
nonlimber_accuracyboost = 1.0

# n(z) photo-z conventions (mirror the likelihood yaml keys):
# interpolation 0 = cspline, 1 = linear, 2+ = Steffen monotone;
# z column 0 = Z_LOW (left bin edges, the DES Y6 files), 1 = Z_MID
photoz_interpolation_type = 0
photoz_zmid_convention = 0

# C FAST-PT grid of the TATT and nonlinear-bias tables, as a fraction
# of its output table (mirrors the likelihood yaml key
# internal_accuracyboost, 1.0 in likelihood/*.yaml; the C default,
# used when nothing sets it, is 0.5)
internal_accuracyboost = 1.0

# Quadrature level 0 (the coarsest Gauss-Legendre rules) and the largest
# multipole of the C_ell tables behind the real-space transforms
integration_accuracy = 0
lmax = 50000

# 26 logarithmic angular bins from 2.5 to 995.27 arcmin, as in the dataset
ntheta = 26
theta_min_arcmin = 2.5
theta_max_arcmin = 995.2679263837432

## Fiducial parameters

The default values of the wrapper arguments are the `sampler: evaluate:
override` block of `EXAMPLE_EVALUATE1.yaml`; the parameters that block does not
list keep their fixed values of `likelihood/params_lens.yaml` and
`likelihood/params_source.yaml`.

- Cosmology: `As_1e9` $=10^9A_s$, the primordial amplitude; `ns` $=n_s$; `H0` in
  km/s/Mpc; `omegab` $=\Omega_b$; `omegam` $=\Omega_m$, all matter including
  massive neutrinos; `mnu` $=\sum m_\nu$ in eV; `w` $=w_0$ and `w0pwa` $=w_0+w_a$,
  so $w_a=0$ here.
- Source side: the photo-z shifts `DES_DZ_S1..4` move each source $n(z)$ in
  redshift by $\Delta z_i$; with the shear calibration `DES_M1..4` the measured
  shear is $(1+m_i)$ times the true shear. NLA with power-law evolution:
  `DES_A1_1` is $A_1$ at $1+z=1.62$ and `DES_A1_2` its exponent $\eta_1$. The
  YAML flags its value, $A_1=0.769=1/1.3$, for a check of the sign and pivot
  conventions. The TATT terms `DES_A2_1`, `DES_A2_2` and `DES_BTA_1` are fixed to
  zero.
- Lens side: the photo-z shifts `DES_DZ_L1..6` move each lens $n(z)$; with the
  linear galaxy bias `DES_B1_1..6` the lens-galaxy density contrast is $b_1$
  times the matter density contrast on large scales.
- The magnification bias `DES_BMAG_1..6` and the point-mass amplitudes
  `DES_PM1..6` are fixed to zero. The point mass is a term in $\gamma_t$ that
  falls as $1/\theta^2$ and absorbs the unmodeled mass close to the lens
  galaxies.

In [ ]:
# Cosmology: As_1e9 = 10^9 A_s, H0 in km/s/Mpc, mnu = sum of neutrino
# masses in eV; w0pwa = w0 + wa, so wa = w0pwa - w = 0
As_1e9 = 2.1
ns = 0.96
H0 = 67.0
omegab = 0.049
omegam = 0.319
mnu = 0.06
w = -1.0
w0pwa = -1.0
# Source bins 1-4: photo-z shifts Delta z_i and shear calibration m_i
DES_DZ_S1 = 0.0
DES_DZ_S2 = 0.0
DES_DZ_S3 = 0.0
DES_DZ_S4 = 0.0
DES_M1 = -0.00339767
DES_M2 = 0.0064552
DES_M3 = 0.01593935
DES_M4 = 0.00169811
# NLA: (A1, eta1) = (DES_A1_1, DES_A1_2). The TATT terms (A2, eta2) =
# (DES_A2_1, DES_A2_2) and b_TA = DES_BTA_1 are fixed to zero in
# likelihood/params_source.yaml; NLA reads none of them.
DES_A1_1 = 0.7692307692307692
DES_A1_2 = -1.0
DES_A2_1 = 0.0
DES_A2_2 = 0.0
DES_BTA_1 = 0.0
# Lens bins 1-6: photo-z shifts Delta z_i and linear galaxy bias b1
DES_DZ_L1 = 0.0
DES_DZ_L2 = 0.0
DES_DZ_L3 = 0.0
DES_DZ_L4 = 0.0
DES_DZ_L5 = 0.0
DES_DZ_L6 = 0.0
DES_B1_1 = 1.54
DES_B1_2 = 1.81
DES_B1_3 = 1.85
DES_B1_4 = 1.76
DES_B1_5 = 1.93
DES_B1_6 = 1.90
# Magnification bias per lens bin, fixed to zero in params_lens.yaml
DES_BMAG_1 = 0.0
DES_BMAG_2 = 0.0
DES_BMAG_3 = 0.0
DES_BMAG_4 = 0.0
DES_BMAG_5 = 0.0
DES_BMAG_6 = 0.0
# Point-mass amplitudes of gamma_t per lens bin, fixed to zero in
# params_lens.yaml (the likelihood's names carry no underscore before the
# bin number)
DES_PM1 = 0.0
DES_PM2 = 0.0
DES_PM3 = 0.0
DES_PM4 = 0.0
DES_PM5 = 0.0
DES_PM6 = 0.0

## Stage 1, CAMB: matter power spectrum, growth and distances

Each wrapper starts with one CAMB run through `cnu.get_camb_cosmology`, shared by
every Cocoa project; `help(cnu.get_camb_cosmology)` prints its documentation.
It returns the ten arrays that `ci.set_cosmology` reads, in CosmoLike's units:

| Returned names | Content |
| --- | --- |
| `log10k_interp_2D`, `z_interp_2D` | The grids of $\log_{10}k$, with $k$ in $h$/Mpc, and of redshift of the power-spectrum tables. |
| `lnPL`, `lnPNL` | $\ln P(k,z)$, linear and nonlinear, with $P$ in (Mpc/$h$)$^3$, each table flattened into one array. |
| `G_growth`, `z_growth` | $G(z)=D(z)(1+z)$, with $D$ the linear growth factor, normalized at the last redshift of the 2D grid, on its own redshift grid. |
| `z_interp_1D`, `chi` | The comoving distance $\chi(z)$ in Mpc/$h$. |
| `omegan2`, `lnPL_cb` | $\Omega_\nu h^2$ and the linear power of cold dark matter plus baryons, read by halo-model quantities; the wrappers pass both, as the likelihood does. |

*Flattened* means the 2D table $(z,k)$ is stored as one 1D array, column after
column (Fortran order), the memory layout the compiled code expects.

Two knobs reach CAMB: it runs with accuracy
`CAMBAccuracyBoost`$\times(1+(\texttt{AccuracyBoost}-1)/3)$, and the redshift and
wavenumber grids grow with `CLAccuracyBoost`$\times$`AccuracyBoost`. The helper
also raises `kmax` and `k_per_logint` with the CAMB accuracy: at the YAML's
`CAMBAccuracyBoost = 1.05`, CAMB samples up to $k=5.75$/Mpc with 11 points per
logarithmic interval.

**Comparison with `EXAMPLE_EVALUATE1.yaml`.** A Cobaya run of the YAML gives
slightly different spectra: the helper's CAMB has one massive neutrino with
$N_{\rm eff}=3.046$, where the YAML has three massive states and
$N_{\rm eff}=3.044$; HMcode's AGN temperature differs as the run settings
describe; and the CAMB $k$ sampling grows with the accuracy setting.

In [ ]:
# get_camb_cosmology is shared by every project's notebooks; call it as
# cnu.get_camb_cosmology (imported above). The CAMB side runs with
# CAMBAccuracyBoost*(1 + (AccuracyBoost - 1)/3); kmax, k_per_logint, the
# chi(z) node count and lens_potential_accuracy grow with that CAMB boost;
# the EuclidEmulator2 boost (non_linear_emul = 1) applies only at z < 10.
# help(cnu.get_camb_cosmology) prints the full documentation.

## Stages 2 to 5: the four wrappers

`C_gs_tomo_limber`, `C_gg_tomo`, `gamma_t` and `w_theta` run, after CAMB, the
same four stages:

2. **Accuracy of the C code.** `ci.set_omp_threads` restores the OpenMP team
   size. `init_ntable_lmax` sets the largest multipole of the spectra tabulated
   for real-space transforms (`lmax` = 50,000 at boost 1, plus 20,000 per unit
   of boost above 1); `init_accuracy_boost` sets the size of CosmoLike's
   interpolation tables and the *quadrature level*, which picks how many points
   evaluate each line-of-sight integral; `init_photoz_conventions` sets how the
   $n(z)$ tables are read. The level passed is
   `CLIntegrationAccuracy` $+\,|3(\text{boost}-1)|$, rounded down, with boost
   $=$ `CLAccuracyBoost`$\times$`AccuracyBoost`: any boost different from 1,
   above or below, raises it.
3. **`ci.set_cosmology`** copies the CAMB tables into CosmoLike, which reads them
   in every later calculation.
4. **Nuisance setters.** On the lens side, `set_nuisance_clustering_photoz`
   receives the lens photo-z shifts and `set_nuisance_bias` the lists `B1`,
   `B2`, `B_MAG`, `B3nl` and `BK`, one entry per lens bin (the last two are zero
   lists). On the source side, in `C_gs_tomo_limber` and `gamma_t`,
   `set_nuisance_shear_photoz` receives the four $\Delta z_i$ and
   `set_nuisance_ia` the IA lists: with `IA_redshift_evolution = 3` only
   `A1[0:2]`, `A2[0:2]` and `B_TA[0]` are read, and the zeros pad each list to
   one entry per source bin, which the setter requires. `reset_bary_struct` switches tabulated baryonic feedback off;
   `init_baryons_contamination` switches it on, and a wrapper refuses it
   together with HMcode's feedback model.
5. **The data-vector calls**:

| Wrapper | Interface call | Returns |
| --- | --- | --- |
| `C_gs_tomo_limber(ell)` | `ci.C_gs_tomo_limber` | $C_\ell^{gs}$ with shape `[n_ell, n_lens, n_source]`, in the Limber approximation. |
| `C_gg_tomo(ell, limber)` | `ci.C_gg_tomo_limber` if `limber == 1`, else `ci.C_gg_tomo` | $C_\ell^{gg}$ with shape `[n_ell, n_lens, n_lens]`, diagonal filled; `ci.C_gg_tomo` replaces Limber by the exact projection below $\ell=150$ and reads it at integer multipoles, so `ell` must hold whole numbers. |
| `gamma_t()` | `ci.w_gammat_tomo` | `(theta, gammat)`, $\theta$ in arcmin and `gammat` with shape `[n_theta, n_lens, n_source]`. |
| `w_theta()` | `ci.w_gg_tomo` | `(theta, w)`, `w` with shape `[n_theta, n_lens, n_lens]`, diagonal filled. |

The spectra are plain, dimensionless $C_\ell$. The *Limber approximation*
evaluates the matter power at $k=(\ell+1/2)/\chi$ along the line of sight
instead of integrating over all wavenumbers; it fails at low multipoles for
narrow redshift bins such as the lens bins. The real-space functions follow
the likelihood's choices, `adopt_limber_gs = 1` and `adopt_limber_gg = 0`:
Limber $\gamma_t$, and $w(\theta)$ exact below $\ell=150$. The angular functions
are averages over each angular bin, and `ci.get_binning_real_space()` returns the
area-weighted bin centers.

The shear calibration $(1+m_i)$ and the point-mass term enter only the
assembled data vector (`get_datavector` in the $\chi^2$ section), never these
four outputs, so the wrappers do not take them.

**Python detail.** The default values in a `def` line (`omegam = omegam`, ...)
are copied once, when the cell runs: rerun the wrapper cells after editing the
fiducial values or the run settings. Names that are not arguments (`mnu`,
`non_linear_emul`, `lmax`, `threads`, the photo-z conventions and `ini`) are read
from the notebook at every call; `ini` is created by the initialization cell
further below, which must run before the first call.

In [ ]:
def C_gs_tomo_limber(ell,
                     omegam = omegam,
                     omegab = omegab,
                     H0 = H0,
                     ns = ns,
                     As_1e9 = As_1e9,
                     w = w,
                     w0pwa = w0pwa,
                     A1 = [DES_A1_1, DES_A1_2, 0, 0],
                     A2 = [DES_A2_1, DES_A2_2, 0, 0],
                     BTA = [DES_BTA_1, 0, 0, 0],
                     shear_photoz_bias = [DES_DZ_S1, DES_DZ_S2, DES_DZ_S3, DES_DZ_S4],
                     lens_photoz_bias = [DES_DZ_L1, DES_DZ_L2, DES_DZ_L3,
                                         DES_DZ_L4, DES_DZ_L5, DES_DZ_L6],
                     galaxy_bias_b1 = [DES_B1_1, DES_B1_2, DES_B1_3,
                                       DES_B1_4, DES_B1_5, DES_B1_6],
                     galaxy_bias_b2 = [0, 0, 0, 0, 0, 0],
                     galaxy_bias_bmag = [DES_BMAG_1, DES_BMAG_2, DES_BMAG_3,
                                         DES_BMAG_4, DES_BMAG_5, DES_BMAG_6],
                     baryon_sims = None,
                     halofit_version = halofit_version,
                     AccuracyBoost = 1.0,
                     kmax = kmax,
                     k_per_logint = k_per_logint,
                     CAMBAccuracyBoost = CAMBAccuracyBoost,
                     CLAccuracyBoost = 1.0,
                     CLIntegrationAccuracy = integration_accuracy):

    # A simulation's power ratio multiplies a gravity-only spectrum; on top
    # of HMcode's feedback term it would count feedback twice. Checked
    # before CAMB runs, so a refused call changes nothing.
    if baryon_sims is not None and halofit_version == "mead2020_feedback":
        raise ValueError(
            f"baryon_sims = {baryon_sims} with halofit_version = "
            f"{halofit_version}: the simulation ratio needs a gravity-only "
            "spectrum; pass halofit_version = 'takahashi'")

    # Stage 1, CAMB: the ten arrays of the CAMB section
    (log10k_interp_2D, z_interp_2D, lnPL, lnPNL, G_growth, z_growth,
     z_interp_1D, chi, omegan2, lnPL_cb) = cnu.get_camb_cosmology(
        omegam = omegam, omegab = omegab, H0 = H0, ns = ns,
        As_1e9 = As_1e9, w = w, w0pwa = w0pwa, mnu = mnu,
        non_linear_emul = non_linear_emul,
        halofit_version = halofit_version,
        AccuracyBoost = AccuracyBoost, kmax = kmax,
        k_per_logint = k_per_logint,
        CAMBAccuracyBoost = CAMBAccuracyBoost,
        CLAccuracyBoost = CLAccuracyBoost)

    # Stage 2, accuracy of the C code. The OpenMP team size comes first: a
    # library loaded in this process can lower it. The quadrature level
    # grows by 3 per unit of |boost - 1|, so a boost below 1 raises it too.
    ci.set_omp_threads(n = threads)
    CLAccuracyBoost = CLAccuracyBoost * AccuracyBoost
    CLIntegrationAccuracy = max(0, CLIntegrationAccuracy + abs(3*(CLAccuracyBoost-1.0)))
    ci.init_ntable_lmax(lmax = int(lmax + 20000*(CLAccuracyBoost-1)))
    ci.init_accuracy_boost(accuracy_boost = CLAccuracyBoost,
                           integration_accuracy = int(CLIntegrationAccuracy))
    ci.init_photoz_conventions(
        interpolation_type = int(photoz_interpolation_type),
        zmid_convention = int(photoz_zmid_convention))

    # Stage 3: copy the CAMB tables into CosmoLike, with omegab, omegan2 and
    # the cold-dark-matter-plus-baryon power lnPL_cb as the likelihood
    # sends them
    ci.set_cosmology(omegam = omegam,
                     omegab = omegab,
                     H0 = H0,
                     log10k_2D = log10k_interp_2D,
                     z_2D = z_interp_2D,
                     lnP_linear = lnPL,
                     lnP_linear_cb = lnPL_cb,
                     lnP_nonlinear = lnPNL,
                     G = G_growth,
                     z_G = z_growth,
                     z_1D = z_interp_1D,
                     chi = chi,
                     omegan2 = omegan2)

    # Stage 4, nuisance parameters: source side (the IA setter reads A1[0:2],
    # A2[0:2] and BTA[0]; zeros pad the lists), then lens side, one entry
    # per lens bin
    ci.set_nuisance_shear_photoz(bias = shear_photoz_bias)
    ci.set_nuisance_ia(A1 = A1, A2 = A2, B_TA = BTA)
    ci.set_nuisance_clustering_photoz(bias = lens_photoz_bias)
    ci.set_nuisance_bias(B1 = galaxy_bias_b1,
                         B2 = galaxy_bias_b2,
                         B_MAG = galaxy_bias_bmag,
                         B3nl = [0, 0, 0, 0, 0, 0],
                         BK = [0, 0, 0, 0, 0, 0])

    # Baryonic feedback off (gravity-only power), or the tabulated power
    # ratio of the named simulation; ini comes from the initialization cell
    if baryon_sims is None:
        ci.reset_bary_struct()
    else:
        ci.init_baryons_contamination(sim = baryon_sims,
            allsims = ini.relativeFileName('all_sims_hdf5_file'))

    # Stage 5: Limber C_ell^gs, [n_ell, n_lens, n_source], plain dimensionless
    return ci.C_gs_tomo_limber(l = ell)

In [ ]:
def C_gg_tomo(ell,
              limber,
              omegam = omegam,
              omegab = omegab,
              H0 = H0,
              ns = ns,
              As_1e9 = As_1e9,
              w = w,
              w0pwa = w0pwa,
              lens_photoz_bias = [DES_DZ_L1, DES_DZ_L2, DES_DZ_L3,
                                  DES_DZ_L4, DES_DZ_L5, DES_DZ_L6],
              galaxy_bias_b1 = [DES_B1_1, DES_B1_2, DES_B1_3,
                                DES_B1_4, DES_B1_5, DES_B1_6],
              galaxy_bias_b2 = [0, 0, 0, 0, 0, 0],
              galaxy_bias_bmag = [DES_BMAG_1, DES_BMAG_2, DES_BMAG_3,
                                  DES_BMAG_4, DES_BMAG_5, DES_BMAG_6],
              baryon_sims = None,
              halofit_version = halofit_version,
              AccuracyBoost = 1.0,
              kmax = kmax,
              k_per_logint = k_per_logint,
              CAMBAccuracyBoost = CAMBAccuracyBoost,
              CLAccuracyBoost = 1.0,
              CLIntegrationAccuracy = integration_accuracy):

    # ci.C_gg_tomo reads the exact projection at integer multipoles and
    # drops any fraction, so whole numbers are required. Both checks run
    # before CAMB, so a refused call changes nothing.
    if limber != 1 and not np.all(np.asarray(ell) == np.floor(ell)):
        raise ValueError(
            "ell holds fractional multipoles: the exact projection reads "
            "whole numbers; build ell as, e.g., np.arange(3., 2500., 3.)")
    # A simulation's power ratio multiplies a gravity-only spectrum
    if baryon_sims is not None and halofit_version == "mead2020_feedback":
        raise ValueError(
            f"baryon_sims = {baryon_sims} with halofit_version = "
            f"{halofit_version}: the simulation ratio needs a gravity-only "
            "spectrum; pass halofit_version = 'takahashi'")

    # Stage 1, CAMB
    (log10k_interp_2D, z_interp_2D, lnPL, lnPNL, G_growth, z_growth,
     z_interp_1D, chi, omegan2, lnPL_cb) = cnu.get_camb_cosmology(
        omegam = omegam, omegab = omegab, H0 = H0, ns = ns,
        As_1e9 = As_1e9, w = w, w0pwa = w0pwa, mnu = mnu,
        non_linear_emul = non_linear_emul,
        halofit_version = halofit_version,
        AccuracyBoost = AccuracyBoost, kmax = kmax,
        k_per_logint = k_per_logint,
        CAMBAccuracyBoost = CAMBAccuracyBoost,
        CLAccuracyBoost = CLAccuracyBoost)

    # Stage 2, accuracy of the C code, as in C_gs_tomo_limber
    ci.set_omp_threads(n = threads)
    CLAccuracyBoost = CLAccuracyBoost * AccuracyBoost
    CLIntegrationAccuracy = max(0, CLIntegrationAccuracy + abs(3*(CLAccuracyBoost-1.0)))
    ci.init_ntable_lmax(lmax = int(lmax + 20000*(CLAccuracyBoost-1)))
    ci.init_accuracy_boost(accuracy_boost = CLAccuracyBoost,
                           integration_accuracy = int(CLIntegrationAccuracy))
    ci.init_photoz_conventions(
        interpolation_type = int(photoz_interpolation_type),
        zmid_convention = int(photoz_zmid_convention))

    # Stage 3: copy the CAMB tables into CosmoLike
    ci.set_cosmology(omegam = omegam,
                     omegab = omegab,
                     H0 = H0,
                     log10k_2D = log10k_interp_2D,
                     z_2D = z_interp_2D,
                     lnP_linear = lnPL,
                     lnP_linear_cb = lnPL_cb,
                     lnP_nonlinear = lnPNL,
                     G = G_growth,
                     z_G = z_growth,
                     z_1D = z_interp_1D,
                     chi = chi,
                     omegan2 = omegan2)

    # Stage 4: lens nuisance parameters, one entry per lens bin
    ci.set_nuisance_clustering_photoz(bias = lens_photoz_bias)
    ci.set_nuisance_bias(B1 = galaxy_bias_b1,
                         B2 = galaxy_bias_b2,
                         B_MAG = galaxy_bias_bmag,
                         B3nl = [0, 0, 0, 0, 0, 0],
                         BK = [0, 0, 0, 0, 0, 0])

    # Baryonic feedback off (gravity-only power), or the tabulated power
    # ratio of the named simulation; ini comes from the initialization cell
    if baryon_sims is None:
        ci.reset_bary_struct()
    else:
        ci.init_baryons_contamination(sim = baryon_sims,
            allsims = ini.relativeFileName('all_sims_hdf5_file'))

    # Stage 5: C_ell^gg, [n_ell, n_lens, n_lens], diagonal filled. limber = 1:
    # Limber everywhere; otherwise the exact projection below ell = 150
    if limber == 1:
        return ci.C_gg_tomo_limber(l = ell)
    else:
        return ci.C_gg_tomo(l = ell)

In [ ]:
def gamma_t(ntheta = ntheta,
            theta_min_arcmin = theta_min_arcmin,
            theta_max_arcmin = theta_max_arcmin,
            omegam = omegam,
            omegab = omegab,
            H0 = H0,
            ns = ns,
            As_1e9 = As_1e9,
            w = w,
            w0pwa = w0pwa,
            A1 = [DES_A1_1, DES_A1_2, 0, 0],
            A2 = [DES_A2_1, DES_A2_2, 0, 0],
            BTA = [DES_BTA_1, 0, 0, 0],
            shear_photoz_bias = [DES_DZ_S1, DES_DZ_S2, DES_DZ_S3, DES_DZ_S4],
            lens_photoz_bias = [DES_DZ_L1, DES_DZ_L2, DES_DZ_L3,
                                DES_DZ_L4, DES_DZ_L5, DES_DZ_L6],
            galaxy_bias_b1 = [DES_B1_1, DES_B1_2, DES_B1_3,
                              DES_B1_4, DES_B1_5, DES_B1_6],
            galaxy_bias_b2 = [0, 0, 0, 0, 0, 0],
            galaxy_bias_bmag = [DES_BMAG_1, DES_BMAG_2, DES_BMAG_3,
                                DES_BMAG_4, DES_BMAG_5, DES_BMAG_6],
            baryon_sims = None,
            halofit_version = halofit_version,
            AccuracyBoost = 1.0,
            kmax = kmax,
            k_per_logint = k_per_logint,
            CAMBAccuracyBoost = CAMBAccuracyBoost,
            CLAccuracyBoost = 1.0,
            CLIntegrationAccuracy = integration_accuracy):

    # A simulation's power ratio multiplies a gravity-only spectrum
    if baryon_sims is not None and halofit_version == "mead2020_feedback":
        raise ValueError(
            f"baryon_sims = {baryon_sims} with halofit_version = "
            f"{halofit_version}: the simulation ratio needs a gravity-only "
            "spectrum; pass halofit_version = 'takahashi'")

    # Stage 1, CAMB
    (log10k_interp_2D, z_interp_2D, lnPL, lnPNL, G_growth, z_growth,
     z_interp_1D, chi, omegan2, lnPL_cb) = cnu.get_camb_cosmology(
        omegam = omegam, omegab = omegab, H0 = H0, ns = ns,
        As_1e9 = As_1e9, w = w, w0pwa = w0pwa, mnu = mnu,
        non_linear_emul = non_linear_emul,
        halofit_version = halofit_version,
        AccuracyBoost = AccuracyBoost, kmax = kmax,
        k_per_logint = k_per_logint,
        CAMBAccuracyBoost = CAMBAccuracyBoost,
        CLAccuracyBoost = CLAccuracyBoost)

    # Stage 2, accuracy of the C code, as in C_gs_tomo_limber
    ci.set_omp_threads(n = threads)
    CLAccuracyBoost = CLAccuracyBoost * AccuracyBoost
    CLIntegrationAccuracy = max(0, CLIntegrationAccuracy + abs(3*(CLAccuracyBoost-1.0)))
    ci.init_ntable_lmax(lmax = int(lmax + 20000*(CLAccuracyBoost-1)))
    ci.init_accuracy_boost(accuracy_boost = CLAccuracyBoost,
                           integration_accuracy = int(CLIntegrationAccuracy))
    ci.init_photoz_conventions(
        interpolation_type = int(photoz_interpolation_type),
        zmid_convention = int(photoz_zmid_convention))
    # ntheta logarithmic bins in arcmin (int: the binding accepts only a
    # Python int). A new binning gets a new cache key: kernels are rebuilt.
    ci.init_binning(ntheta_bins = int(ntheta),
                    theta_min_arcmin = theta_min_arcmin,
                    theta_max_arcmin = theta_max_arcmin)

    # Stage 3: copy the CAMB tables into CosmoLike
    ci.set_cosmology(omegam = omegam,
                     omegab = omegab,
                     H0 = H0,
                     log10k_2D = log10k_interp_2D,
                     z_2D = z_interp_2D,
                     lnP_linear = lnPL,
                     lnP_linear_cb = lnPL_cb,
                     lnP_nonlinear = lnPNL,
                     G = G_growth,
                     z_G = z_growth,
                     z_1D = z_interp_1D,
                     chi = chi,
                     omegan2 = omegan2)

    # Stage 4: source and lens nuisance parameters, as in C_gs_tomo_limber
    ci.set_nuisance_shear_photoz(bias = shear_photoz_bias)
    ci.set_nuisance_ia(A1 = A1, A2 = A2, B_TA = BTA)
    ci.set_nuisance_clustering_photoz(bias = lens_photoz_bias)
    ci.set_nuisance_bias(B1 = galaxy_bias_b1,
                         B2 = galaxy_bias_b2,
                         B_MAG = galaxy_bias_bmag,
                         B3nl = [0, 0, 0, 0, 0, 0],
                         BK = [0, 0, 0, 0, 0, 0])

    # Baryonic feedback off (gravity-only power), or the tabulated power
    # ratio of the named simulation; ini comes from the initialization cell
    if baryon_sims is None:
        ci.reset_bary_struct()
    else:
        ci.init_baryons_contamination(sim = baryon_sims,
            allsims = ini.relativeFileName('all_sims_hdf5_file'))

    # Stage 5: (theta in arcmin, gamma_t [n_theta, n_lens, n_source])
    return (ci.get_binning_real_space(), ci.w_gammat_tomo())

In [ ]:
def w_theta(ntheta = ntheta,
            theta_min_arcmin = theta_min_arcmin,
            theta_max_arcmin = theta_max_arcmin,
            omegam = omegam,
            omegab = omegab,
            H0 = H0,
            ns = ns,
            As_1e9 = As_1e9,
            w = w,
            w0pwa = w0pwa,
            lens_photoz_bias = [DES_DZ_L1, DES_DZ_L2, DES_DZ_L3,
                                DES_DZ_L4, DES_DZ_L5, DES_DZ_L6],
            galaxy_bias_b1 = [DES_B1_1, DES_B1_2, DES_B1_3,
                              DES_B1_4, DES_B1_5, DES_B1_6],
            galaxy_bias_b2 = [0, 0, 0, 0, 0, 0],
            galaxy_bias_bmag = [DES_BMAG_1, DES_BMAG_2, DES_BMAG_3,
                                DES_BMAG_4, DES_BMAG_5, DES_BMAG_6],
            baryon_sims = None,
            halofit_version = halofit_version,
            AccuracyBoost = 1.0,
            kmax = kmax,
            k_per_logint = k_per_logint,
            CAMBAccuracyBoost = CAMBAccuracyBoost,
            CLAccuracyBoost = 1.0,
            CLIntegrationAccuracy = integration_accuracy):

    # A simulation's power ratio multiplies a gravity-only spectrum
    if baryon_sims is not None and halofit_version == "mead2020_feedback":
        raise ValueError(
            f"baryon_sims = {baryon_sims} with halofit_version = "
            f"{halofit_version}: the simulation ratio needs a gravity-only "
            "spectrum; pass halofit_version = 'takahashi'")

    # Stage 1, CAMB
    (log10k_interp_2D, z_interp_2D, lnPL, lnPNL, G_growth, z_growth,
     z_interp_1D, chi, omegan2, lnPL_cb) = cnu.get_camb_cosmology(
        omegam = omegam, omegab = omegab, H0 = H0, ns = ns,
        As_1e9 = As_1e9, w = w, w0pwa = w0pwa, mnu = mnu,
        non_linear_emul = non_linear_emul,
        halofit_version = halofit_version,
        AccuracyBoost = AccuracyBoost, kmax = kmax,
        k_per_logint = k_per_logint,
        CAMBAccuracyBoost = CAMBAccuracyBoost,
        CLAccuracyBoost = CLAccuracyBoost)

    # Stage 2, accuracy of the C code, as in C_gs_tomo_limber
    ci.set_omp_threads(n = threads)
    CLAccuracyBoost = CLAccuracyBoost * AccuracyBoost
    CLIntegrationAccuracy = max(0, CLIntegrationAccuracy + abs(3*(CLAccuracyBoost-1.0)))
    ci.init_ntable_lmax(lmax = int(lmax + 20000*(CLAccuracyBoost-1)))
    ci.init_accuracy_boost(accuracy_boost = CLAccuracyBoost,
                           integration_accuracy = int(CLIntegrationAccuracy))
    ci.init_photoz_conventions(
        interpolation_type = int(photoz_interpolation_type),
        zmid_convention = int(photoz_zmid_convention))
    # ntheta logarithmic bins in arcmin, as in gamma_t
    ci.init_binning(ntheta_bins = int(ntheta),
                    theta_min_arcmin = theta_min_arcmin,
                    theta_max_arcmin = theta_max_arcmin)

    # Stage 3: copy the CAMB tables into CosmoLike
    ci.set_cosmology(omegam = omegam,
                     omegab = omegab,
                     H0 = H0,
                     log10k_2D = log10k_interp_2D,
                     z_2D = z_interp_2D,
                     lnP_linear = lnPL,
                     lnP_linear_cb = lnPL_cb,
                     lnP_nonlinear = lnPNL,
                     G = G_growth,
                     z_G = z_growth,
                     z_1D = z_interp_1D,
                     chi = chi,
                     omegan2 = omegan2)

    # Stage 4: lens nuisance parameters, one entry per lens bin
    ci.set_nuisance_clustering_photoz(bias = lens_photoz_bias)
    ci.set_nuisance_bias(B1 = galaxy_bias_b1,
                         B2 = galaxy_bias_b2,
                         B_MAG = galaxy_bias_bmag,
                         B3nl = [0, 0, 0, 0, 0, 0],
                         BK = [0, 0, 0, 0, 0, 0])

    # Baryonic feedback off (gravity-only power), or the tabulated power
    # ratio of the named simulation; ini comes from the initialization cell
    if baryon_sims is None:
        ci.reset_bary_struct()
    else:
        ci.init_baryons_contamination(sim = baryon_sims,
            allsims = ini.relativeFileName('all_sims_hdf5_file'))

    # Stage 5: (theta in arcmin, w [n_theta, n_lens, n_lens], diagonal filled)
    return (ci.get_binning_real_space(), ci.w_gg_tomo())

## The shared plotting functions

- `cnu.plot_C_gs_tomo_limber` and `cnu.plot_gammat_tomo_limber` draw one panel per
  (lens, source) pair: rows are source bins, columns lens bins, and the panel
  label reads (lens, source).
- `cnu.plot_C_gg_tomo` and `cnu.plot_wtheta_tomo` draw one row of panels, one per
  lens bin, from the diagonal of the arrays.

They work in one of two modes:

- **Absolute**, without a reference argument: each panel has its own logarithmic
  y axis. The spectrum plotters draw $C_\ell$ itself and do not take its
  absolute value, so a negative value would vanish from a logarithmic axis; the
  real-space plotters draw $|\gamma_t|$ and $|w|$. `ylim = [a, b]` multiplies each
  panel's minimum by $a$ and its maximum by $b$.
- **Reference**, with `C_gs_ref=`, `gammat_ref=`, `C_gg_ref=` or
  `theta_wtheta_ref=`: each panel shows value/reference $-\,1$ on one shared linear
  band. `ylim` is then written around 1: `(0.981, 1.019)` draws the band from
  $-0.019$ to $+0.019$.

A list `param` of swept values colors the curves and the colorbar; `legend` names
the curves instead and, by default, sits above the panels. `thetashow` sets the
horizontal range of the real-space panels in arcmin.
`help(cnu.plot_gammat_tomo_limber)` lists every option.

In [ ]:
# plot_C_gs_tomo_limber is shared by every project; call it as
# cnu.plot_C_gs_tomo_limber (imported above). Rows are source bins,
# columns lens bins; help(cnu.plot_C_gs_tomo_limber) prints every argument.

In [ ]:
# plot_C_gg_tomo is shared by every project; call it as cnu.plot_C_gg_tomo
# (imported above). help(cnu.plot_C_gg_tomo) prints every argument.

In [ ]:
# plot_gammat_tomo_limber is shared by every project; call it as
# cnu.plot_gammat_tomo_limber (imported above). Rows are source bins,
# columns lens bins; thetashow sets the horizontal range in arcmin.

In [ ]:
# plot_wtheta_tomo is shared by every project; call it as
# cnu.plot_wtheta_tomo (imported above). help(cnu.plot_wtheta_tomo) prints
# every argument.

## Initialize CosmoLike once

The next cell configures the compiled module before the first wrapper call, in
the order of the likelihood's `initialize`
(`likelihood/_cosmolike_prototype_base.py`); the wrappers then change only
cosmology, nuisance parameters and accuracy.

- `ini` reads `DESY6.dataset`; `relativeFileName(key)` returns the file named
  under `key`, with the dataset's folder prepended.
- `initial_setup` resets every CosmoLike setting to its default.
- `init_fpt_internal_boost` comes before the first `init_accuracy_boost` call,
  because that call scales the value it finds then.
- `init_nonlimber_accuracy_boost`, `init_adopt_limber_gs` and
  `init_adopt_limber_gg` set the distance grid of the exact projection and
  where it replaces Limber. `init_include_HOD_GX(0)` and `init_include_halo_IA(0)`
  keep the perturbative galaxy bias and the `init_IA` model rather than their
  halo-model alternatives.
- `init_cosmo_runmode(is_linear=False)` selects the nonlinear matter power.
- `init_redshift_distributions_from_files` reads the lens and source $n(z)$
  tables: a redshift column, then one column per tomographic bin.
- `init_IA` selects the IA model, its redshift evolution and the FAST-PT code;
  `init_bias` the redshift model of the six bias terms.

The probe selection, the dataset's binning and its data files wait for the
$\chi^2$ section. The `int(...)` conversions are needed because several
bindings accept only a Python integer, not a float or a NumPy integer.

In [ ]:
# Init Cosmolike
# relativeFileName(key) returns the file named under key in the dataset
# file, with the dataset's folder prepended
ini = IniFile(os.path.normpath(os.path.join(path, data_file)))

lens_file = ini.relativeFileName('nz_lens_file')
source_file = ini.relativeFileName('nz_source_file')
lens_ntomo = ini.int("lens_ntomo")
source_ntomo = ini.int("source_ntomo")

# Reset every CosmoLike setting to its default value
ci.initial_setup()
ci.set_omp_threads(n = threads)

ci.init_photoz_conventions(
    interpolation_type = int(photoz_interpolation_type),
    zmid_convention = int(photoz_zmid_convention))

# set before the first init_accuracy_boost, as the likelihood does:
# init_accuracy_boost scales this fraction from the value it finds at
# its first call in the process
ci.init_fpt_internal_boost(internal_boost = float(internal_accuracyboost))

# Distance grid of the exact projection, and where it replaces Limber
ci.init_nonlimber_accuracy_boost(nonlimber_boost = float(nonlimber_accuracyboost))
ci.init_adopt_limber_gs(adopt_limber_gs = int(adopt_limber_gs))
ci.init_adopt_limber_gg(adopt_limber_gg = int(adopt_limber_gg))
# 0: perturbative galaxy bias and the init_IA model, not the halo-model
# alternatives (include_HOD_GX: 0 and include_halo_IA: 0 in the YAML)
ci.init_include_HOD_GX(include_HOD_GX = 0)
ci.init_include_halo_IA(include_halo_IA = 0)

# Baseline table sizes and quadrature level; every wrapper call sets its own
ci.init_ntable_lmax(lmax = int(lmax))
ci.init_accuracy_boost(accuracy_boost = 1.0,
                       integration_accuracy = int(integration_accuracy))

# False: use the nonlinear matter power spectrum
ci.init_cosmo_runmode(is_linear = False)

# int(): these bindings refuse floats and NumPy integers
ci.init_redshift_distributions_from_files(
      lens_multihisto_file = lens_file,
      lens_ntomo = int(lens_ntomo),
      source_multihisto_file = source_file,
      source_ntomo = int(source_ntomo))

ci.init_IA(ia_model = int(IA_model),
           ia_redshift_evolution = int(IA_redshift_evolution),
           ia_code = int(IA_code))

# Codes for (b1, b2, bs2, b3, bmag, bK): one constant amplitude per lens bin
ci.init_bias(bias_model = bias_model)

## How $\Omega_m$ and $n_s$ change $C_\ell^{gs}$

The first sweep varies $\Omega_m$ from 0.20 to 0.38 in steps of 0.02 and draws the
absolute spectra; the plotter draws the signed $C_\ell^{gs}$ on logarithmic axes,
so the cell passes `np.abs(C_gs)`. The second sweep varies $n_s$ from 0.85 to
1.09 and draws each spectrum relative to the fiducial one,
$C_\ell/C_\ell^{\rm fid}-1$. Both use 825 multipoles from 25 to 2497. Panels whose
lens bin lies behind the source bin carry little lensing signal.

In [ ]:
# 825 multipoles from 25 to 2497; float endpoints make np.arange return
# floats, the type the bindings read as multipoles
ell = np.arange(25., 2500., 3.)
param = np.arange(0.2, 0.4, 0.02)
C_gs = []
for x in param:
    C_gs.append(C_gs_tomo_limber(ell = ell, omegam = x))

In [ ]:
# Absolute mode on logarithmic axes: np.abs keeps a negative C_gs visible.
# Applied to the list, it returns one array [n_param, n_ell, n_lens, n_source].
cnu.plot_C_gs_tomo_limber(ell = ell,
                          C_gs = np.abs(C_gs),
                          param = param,
                          colorbarlabel = r"$\Omega_m$",
                          bintextpos = [0.8, 0.875],
                          ylim = [0.75, 1.3],
                          lmin = ell[0],
                          lmax = ell[len(ell)-1],
                          figsize = (21, 13),
                          bintextsize = 20,
                          yaxislabelsize = 17,
                          yaxisticklabelsize = 14,
                          xaxisticklabelsize = 20)

In [ ]:
# 825 multipoles from 25 to 2497, as above
ell = np.arange(25., 2500., 3.)
param = np.arange(0.85, 1.1, 0.02)
C_gs = []
for x in param:
    C_gs.append(C_gs_tomo_limber(ell = ell, ns = x))

# The fiducial spectrum, the reference of the ratios
ref = C_gs_tomo_limber(ell = ell)

In [ ]:
# Reference mode: C/C_ref - 1; ylim (0.64, 1.45) is the band -36% to +45%
cnu.plot_C_gs_tomo_limber(ell = ell,
                          C_gs = C_gs,
                          C_gs_ref = ref,
                          param = param,
                          colorbarlabel = r"$n_s$",
                          bintextpos = [0.15, 0.875],
                          ylim = [0.64, 1.45],
                          lmin = ell[0],
                          lmax = ell[len(ell)-1],
                          figsize = (20, 12),
                          bintextsize = 20,
                          yaxislabelsize = 17,
                          yaxisticklabelsize = 14,
                          xaxisticklabelsize = 20)

## How $\Omega_m$ and $A_s$ change $\gamma_t(\theta)$

The *tangential shear* $\gamma_t(\theta)$ is the shear of source galaxies in the
direction tangential to a lens galaxy at angular separation $\theta$, averaged
over all pairs: the real-space counterpart of $C_\ell^{gs}$. It measures the
galaxy–matter cross-correlation, which in this linear-bias model is $b_1$ times
the matter correlation. The first sweep varies $\Omega_m$ from 0.20 to 0.38;
each entry of `gammat_theta` is the pair `(theta, gammat)` returned by
`gamma_t`, and the absolute panels draw $|\gamma_t|$ over the full angular range,
2.5 to 995.27 arcmin. All 24 pairs are computed, so pairs with little redshift
overlap show small signals.

The second sweep evaluates $10^9A_s$ = 1.9, 2.1 and 2.3, the fiducial and
$\pm0.2$ around it, and draws each curve relative to the fiducial. In linear
theory $\gamma_t$ scales as $A_s$, so the large-angle ratios sit near
$1.9/2.1-1$ and $2.3/2.1-1$, about −9.5% and +9.5%; at small angles the
nonlinear power responds more strongly.

In [ ]:
param = np.arange(0.2, 0.4, 0.02)
gammat_theta = []
for x in param:
    # Each entry is the pair (theta, gamma_t)
    gammat_theta.append(gamma_t(omegam = x))

In [ ]:
# Absolute mode: |gamma_t| on logarithmic axes; ylim multiplies each
# panel's minimum by 0.75 and its maximum by 1.3
cnu.plot_gammat_tomo_limber(theta_gammat = gammat_theta,
                            gammat_ref = None,
                            param = param,
                            colorbarlabel = r"$\Omega_m$",
                            thetashow = [theta_min_arcmin, theta_max_arcmin],
                            bintextpos = [0.8, 0.875],
                            ylim = [0.75, 1.3],
                            figsize = (21, 13),
                            bintextsize = 20,
                            yaxislabelsize = 17,
                            yaxisticklabelsize = 14,
                            xaxisticklabelsize = 20)

In [ ]:
# 10^9 A_s = 1.9, 2.1 and 2.3; 2.1 is the YAML fiducial, so the middle
# curve coincides with the reference
param = np.array([1.9, 2.1, 2.3])
gammat_theta = []
for x in param:
    gammat_theta.append(gamma_t(As_1e9 = x))
gammat_theta_ref = gamma_t()

In [ ]:
# Reference mode: gamma_t/gamma_t,fid - 1; ylim (0.86, 1.14) is the band
# -14% to +14%
cnu.plot_gammat_tomo_limber(theta_gammat = gammat_theta,
                            gammat_ref = gammat_theta_ref,
                            param = param,
                            colorbarlabel = r"$10^9 A_s$",
                            thetashow = [theta_min_arcmin, theta_max_arcmin],
                            bintextpos = [0.85, 0.2],
                            ylim = (0.86, 1.14),
                            figsize = (21, 13),
                            bintextsize = 20,
                            yaxislabelsize = 17,
                            yaxisticklabelsize = 14,
                            xaxisticklabelsize = 20)

## Changing the angular binning without restarting the kernel

The *kernel* is the Python process that runs the cells. CosmoLike keeps
expensive intermediate results, such as the bin-averaged kernels that turn
$C_\ell$ into $\gamma_t(\theta)$, in static arrays: C memory that survives between
calls. Each cached table carries a key; `ci.init_binning` assigns a new key
whenever the number of bins or the angular range changes, so the next call
rebuilds the kernels instead of reusing those of the previous binning.

The cell computes $\gamma_t$ with 9, 12, 17 and 44 bins between 2.5 and 995.27
arcmin, and the figure marks each binning with its own symbol. The points of all
binnings should follow one smooth curve, wide bins deviating slightly because
each point averages over its bin; a stale cache would instead place the
previous binning's values at the new angles.

In [ ]:
nthetas = [9, 12, 17, 44]
gammat_theta = []
for x in nthetas:
    # A new binning on every call: CosmoLike rebuilds its angular kernels
    gammat_theta.append(gamma_t(ntheta = x))

In [ ]:
cnu.plot_gammat_tomo_limber(theta_gammat = gammat_theta,
                            marker = ['s', 'o', ">", "<"],
                            thetashow = [theta_min_arcmin, theta_max_arcmin],
                            ylim = [0.75, 1.3],
                            bintextpos = [0.8, 0.875],
                            figsize = (21, 13),
                            bintextsize = 20,
                            yaxislabelsize = 17,
                            yaxisticklabelsize = 14,
                            xaxisticklabelsize = 20)

## Baryonic feedback from tabulated simulations in $C_\ell^{gs}$ and $\gamma_t(\theta)$

Gas physics (cooling, star formation, energy injected by supernovae and active
galactic nuclei) moves matter inside and around halos and changes the matter
power spectrum at small scales. `ci.init_baryons_contamination(sim, allsims)`
multiplies CosmoLike's nonlinear matter power by the ratio of the power spectrum
of one hydrodynamical simulation to that of its dark-matter-only counterpart,
tabulated in the HDF5 file (a binary table format) `baryons_logPkR.h5` named in
the dataset. The ratio is a fixed table: it does not change with cosmology or
with any feedback parameter.

| Label | Simulation |
| --- | --- |
| `TNG100-1` | IllustrisTNG, TNG100 box |
| `HzAGN-1` | Horizon-AGN |
| `mb2-1` | MassiveBlack-II |
| `illustris-1` | Illustris |
| `eagle-1` | EAGLE |
| `owls_AGN-1` | cosmo-OWLS AGN, heating temperature $10^{8.0}$ K |
| `owls_AGN-2` | cosmo-OWLS AGN, heating temperature $10^{8.5}$ K |
| `owls_AGN-3` | cosmo-OWLS AGN, heating temperature $10^{8.7}$ K |
| `BAHAMAS-2` | BAHAMAS, heating temperature $10^{7.6}$ K |

The cells compute these nine scenarios and the reference without feedback, all
on the gravity-only Takahashi spectrum (`halofit_version = "takahashi"`), then
draw each scenario relative to the reference, named by a legend above the
panels instead of a colorbar. The real-space curves use 40 angular bins, for
smoother lines. The `bfmt` section at the end of the notebook computes the same
kind of ratio from parametric emulators instead of fixed tables.

In [ ]:
# 750 multipoles from 3 to 7493 (float endpoints: np.arange returns floats)
ell = np.arange(3., 7500., 10.)
# Simulation names in the HDF5 table baryons_logPkR.h5
param = ("TNG100-1", "HzAGN-1", "mb2-1", "illustris-1", "eagle-1",
         "owls_AGN-1", "owls_AGN-2", "owls_AGN-3", "BAHAMAS-2")

# A simulation ratio multiplies a gravity-only spectrum, and the reference
# uses the same one: every call passes the Takahashi Halofit
C_gs = []
for x in param:
    C_gs.append(C_gs_tomo_limber(ell = ell, baryon_sims = x,
                                 halofit_version = "takahashi"))
C_gs_ref = C_gs_tomo_limber(ell = ell, halofit_version = "takahashi")

gammat_theta = []
for x in param:
    # 40 angular bins draw smoother curves than the 26 of the data vector
    gammat_theta.append(gamma_t(baryon_sims = x, ntheta = 40,
                                halofit_version = "takahashi"))
gammat_theta_ref = gamma_t(ntheta = 40, halofit_version = "takahashi")

In [ ]:
# ylim (0.61, 1.07): the band -39% to +7% of C/C_ref - 1. tab10 gives nine
# distinct dark colors; the linestyles separate curves of similar color.
cnu.plot_C_gs_tomo_limber(ell = ell,
                          C_gs = C_gs,
                          C_gs_ref = C_gs_ref,
                          lmin = 50,
                          lmax = ell[len(ell)-1],
                          cmap = "tab10",
                          bintextpos = [0.15, 0.2],
                          ylim = [0.61, 1.07],
                          legend = param,
                          legendfontsize = 18,
                          linewidth = [2.0],
                          linestyle = ['solid', 'dashed', 'dashdot', 'dotted'],
                          figsize = (18, 12),
                          bintextsize = 20,
                          yaxislabelsize = 17,
                          yaxisticklabelsize = 14,
                          xaxisticklabelsize = 20)

In [ ]:
# ylim (0.63, 1.07): the band -37% to +7% of gamma_t/gamma_t,ref - 1
cnu.plot_gammat_tomo_limber(theta_gammat = gammat_theta,
                            gammat_ref = gammat_theta_ref,
                            cmap = "tab10",
                            thetashow = [theta_min_arcmin, theta_max_arcmin],
                            bintextpos = [0.85, 0.2],
                            ylim = [0.63, 1.07],
                            legend = param,
                            legendfontsize = 18,
                            linewidth = [2.0],
                            linestyle = ['solid', 'dashed', 'dashdot', 'dotted'],
                            figsize = (18, 13),
                            bintextsize = 20,
                            yaxislabelsize = 17,
                            yaxisticklabelsize = 14,
                            xaxisticklabelsize = 20)

## Accuracy settings: how `AccuracyBoost` changes $C_\ell^{gs}$ and $\gamma_t(\theta)$

`AccuracyBoost` refines three things together: CAMB runs with accuracy
`CAMBAccuracyBoost`$\times(1+(\texttt{AccuracyBoost}-1)/3)$, the CosmoLike tables
and the CAMB grids grow with `AccuracyBoost`, and the wrapper raises the
quadrature level by $|3(\text{boost}-1)|$, rounded down, so a boost below 1
raises it too.

The cell sweeps `AccuracyBoost` from 0.6 to 1.8 in steps of 0.2 and draws each
curve relative to `AccuracyBoost = 1`, on a band of ±1.9%. The $\chi^2$
sections below express such differences in units of the forecast errors.

In [ ]:
# 283 multipoles from 10 to 8470 (float endpoints: np.arange returns floats)
ell = np.arange(10., 8500., 30.)

# AccuracyBoost 0.6 to 1.8 refines CAMB, the CosmoLike tables and the
# quadrature level together
param = np.arange(0.6, 2.0, 0.2)

C_gs = []
for x in param:
    C_gs.append(C_gs_tomo_limber(ell = ell, AccuracyBoost = x))
# The AccuracyBoost = 1 spectrum, the reference of the ratios
C_gs_ref = C_gs_tomo_limber(ell = ell)

gammat_theta = []
for x in param:
    gammat_theta.append(gamma_t(ntheta = 40, AccuracyBoost = x))
gammat_theta_ref = gamma_t(ntheta = 40, AccuracyBoost = 1.0)

In [ ]:
# ylim (0.981, 1.019): the band -1.9% to +1.9% of C/C_ref - 1
cnu.plot_C_gs_tomo_limber(ell = ell,
                          C_gs = C_gs,
                          C_gs_ref = C_gs_ref,
                          param = param,
                          lmin = 10,
                          lmax = 8000,
                          ylim = (0.981, 1.019),
                          bintextpos = [0.85, 0.2],
                          colorbarlabel = "Accuracy Boost",
                          linewidth = [1.0, 1.3, 1.6, 1.9],
                          linestyle = ['solid', 'dashed', 'dashdot', 'dotted'],
                          figsize = (20, 13),
                          bintextsize = 20,
                          yaxislabelsize = 17,
                          yaxisticklabelsize = 14,
                          xaxisticklabelsize = 20)

In [ ]:
cnu.plot_gammat_tomo_limber(theta_gammat = gammat_theta,
                            gammat_ref = gammat_theta_ref,
                            thetashow = [theta_min_arcmin, theta_max_arcmin],
                            bintextpos = [0.85, 0.2],
                            ylim = (0.981, 1.019),
                            param = param,
                            colorbarlabel = "Accuracy Boost",
                            linewidth = [1.0, 1.3, 1.6, 1.9],
                            linestyle = ['solid', 'dashed', 'dashdot', 'dotted'],
                            figsize = (20, 13),
                            bintextsize = 20,
                            yaxislabelsize = 17,
                            yaxisticklabelsize = 14,
                            xaxisticklabelsize = 20)

## How $\Omega_m$ changes $C_\ell^{gg}$, with and without the Limber approximation

`C_gg_tomo(ell, limber=1)` uses the Limber approximation at every multipole;
`limber=0` computes the exact projection below $\ell=150$, an integral over all
wavenumbers with spherical Bessel functions evaluated with FFTLog (a fast
Fourier transform on a logarithmic grid), and keeps Limber above. The exact
projection is $C_\ell=C_\ell^{\rm exact}(P_{\rm lin})+C_\ell^{\rm Limber}(P_{\rm NL})-C_\ell^{\rm Limber}(P_{\rm lin})$:
the nonlinear correction comes from Limber, where it is accurate. It includes
redshift-space distortions (RSD), the line-of-sight displacement of galaxies by
their peculiar velocities, and each lens bin switches back to Limber once the
two agree within 1%.

The first cell sweeps $\Omega_m$ from 0.20 to 0.38 for both and draws the absolute
spectra, one panel per lens bin, at 833 multipoles from 3 to 2499. The next cells
compute the ratio exact/Limber at every multipole from 3 to 249 and draw it up to
$\ell=120$: narrow lens bins make the Limber approximation fail at low
multipoles.

In [ ]:
# 833 multipoles from 3 to 2499; the exact projection needs whole numbers
ell = np.arange(3., 2500., 3.)
param = np.arange(0.2, 0.4, 0.02)
C_gg_L = []
C_gg_NL = []
for x in param:
    # limber=1: Limber at every multipole; limber=0: exact below ell = 150
    tmp1 = C_gg_tomo(ell = ell, limber = 1, omegam = x)
    tmp2 = C_gg_tomo(ell = ell, limber = 0, omegam = x)
    C_gg_L.append(tmp1)
    C_gg_NL.append(tmp2)

In [ ]:
cnu.plot_C_gg_tomo(ell = ell,
                   C_gg = C_gg_L,
                   param = param,
                   colorbarlabel = r"$\Omega_m$",
                   bintextpos = [0.8, 0.875],
                   ylim = [0.75, 1.3],
                   lmin = ell[0],
                   lmax = ell[len(ell)-1],
                   overwriteylabel = r"$C_{\ell,{\rm L}}^{gg}$",
                   figsize = (21, 13./5),
                   bintextsize = 20,
                   yaxislabelsize = 17,
                   yaxisticklabelsize = 14,
                   xaxisticklabelsize = 20)
cnu.plot_C_gg_tomo(ell = ell,
                   C_gg = C_gg_NL,
                   param = param,
                   colorbarlabel = r"$\Omega_m$",
                   bintextpos = [0.8, 0.875],
                   ylim = [0.75, 1.3],
                   lmin = ell[0],
                   lmax = ell[len(ell)-1],
                   overwriteylabel = r"$C_{\ell,{\rm NL}}^{gg}$",
                   figsize = (21, 13./5),
                   bintextsize = 20,
                   yaxislabelsize = 17,
                   yaxisticklabelsize = 14,
                   xaxisticklabelsize = 20)

In [ ]:
# Every integer multipole from 3 to 249
ell = np.arange(3., 250., 1.)
param = np.arange(0.2, 0.4, 0.02)
ratio = []
for x in param:
    tmp1 = C_gg_tomo(ell = ell, limber = 1, omegam = x)
    tmp2 = C_gg_tomo(ell = ell, limber = 0, omegam = x)
    # Exact over Limber: equal to 1 from ell = 150 on, where both use Limber
    ratio.append(tmp2/tmp1)

In [ ]:
# The ratio itself, drawn in absolute mode on a forced linear axis: ylim
# multiplies its minimum by 0.95 and its maximum by 1.05
cnu.plot_C_gg_tomo(ell = ell,
                   C_gg = ratio,
                   param = param,
                   colorbarlabel = r"$\Omega_m$",
                   bintextpos = [0.8, 0.875],
                   ylim = [0.95, 1.05],
                   lmin = 3,
                   lmax = 120,
                   forcelinearyscale = True,
                   overwriteylabel = r"$C_{\ell,{\rm NL}}^{gg}/C_{\ell,{\rm L}}^{gg}$",
                   forcelinearxscale = False,
                   marker = None,
                   figsize = (21, 13./5),
                   bintextsize = 20,
                   yaxislabelsize = 17,
                   yaxisticklabelsize = 14,
                   xaxisticklabelsize = 20)

## How $\Omega_m$ changes $w(\theta)$

The angular correlation function $w(\theta)$ is the excess probability, over a
random distribution, of finding two lens galaxies of the same bin at angular
separation $\theta$: the real-space counterpart of $C_\ell^{gg}$, computed with
the exact projection below $\ell=150$. In the linear-bias model it follows $b_1^2$
times the projected matter correlation. The sweep varies $\Omega_m$ from 0.20 to
0.38; the absolute panels draw $|w|$, one per lens bin, so a dip toward zero at
large $\theta$ can mark a sign change of $w$.

In [ ]:
param = np.arange(0.2, 0.4, 0.02)
wtheta_theta = []
for x in param:
    # Each entry is the pair (theta, w)
    wtheta_theta.append(w_theta(omegam = x))

In [ ]:
# Absolute mode: |w| on logarithmic axes, one panel per lens bin
cnu.plot_wtheta_tomo(theta_wtheta = wtheta_theta,
                     theta_wtheta_ref = None,
                     param = param,
                     colorbarlabel = r"$\Omega_m$",
                     thetashow = [theta_min_arcmin, theta_max_arcmin],
                     bintextpos = [0.8, 0.875],
                     ylim = [0.75, 1.3],
                     figsize = (21, 13./5),
                     bintextsize = 20,
                     yaxislabelsize = 17,
                     yaxisticklabelsize = 14,
                     xaxisticklabelsize = 20)

## Changing the binning of $w(\theta)$ without restarting the kernel

As for $\gamma_t$ above: `ci.init_binning` gives the cached bin-averaged kernels a
new key whenever the binning changes, so the calls with 9, 12, 17 and 44 bins
each rebuild them. The points of all binnings should follow one smooth curve.

In [ ]:
nthetas = [9, 12, 17, 44]
wtheta_theta = []
for x in nthetas:
    # A new binning on every call: CosmoLike rebuilds its angular kernels
    wtheta_theta.append(w_theta(ntheta = x))

In [ ]:
cnu.plot_wtheta_tomo(theta_wtheta = wtheta_theta,
                     marker = ['s', 'o', ">", "<"],
                     thetashow = [theta_min_arcmin, theta_max_arcmin],
                     ylim = [0.75, 1.3],
                     bintextpos = [0.8, 0.875],
                     figsize = (21, 13./5),
                     bintextsize = 20,
                     yaxislabelsize = 17,
                     yaxisticklabelsize = 14,
                     xaxisticklabelsize = 20)

## Baryonic feedback from tabulated simulations in $C_\ell^{gg}$ and $w(\theta)$

The nine hydrodynamical scenarios of the galaxy–galaxy lensing section, now for
clustering, again on the gravity-only Takahashi spectrum. `limber=0` selects the
exact projection below $\ell=150$. Each scenario is drawn relative to the
reference without feedback.

In [ ]:
# 750 multipoles from 3 to 7493, whole numbers for the exact projection
ell = np.arange(3., 7500., 10.)
# Simulation names in the HDF5 table baryons_logPkR.h5
param = ("TNG100-1", "HzAGN-1", "mb2-1", "illustris-1", "eagle-1",
         "owls_AGN-1", "owls_AGN-2", "owls_AGN-3", "BAHAMAS-2")

C_gg = []
for x in param:
    # limber=0: the exact projection below ell = 150
    C_gg.append(C_gg_tomo(ell = ell, limber = 0, baryon_sims = x,
                          halofit_version = "takahashi"))
C_gg_ref = C_gg_tomo(ell = ell, limber = 0, halofit_version = "takahashi")

wtheta_theta = []
for x in param:
    # 40 angular bins draw smoother curves than the 26 of the data vector
    wtheta_theta.append(w_theta(baryon_sims = x, ntheta = 40,
                                halofit_version = "takahashi"))
wtheta_theta_ref = w_theta(ntheta = 40, halofit_version = "takahashi")

In [ ]:
# ylim (0.61, 1.07): the band -39% to +7% of C/C_ref - 1
cnu.plot_C_gg_tomo(ell = ell,
                   C_gg = C_gg,
                   C_gg_ref = C_gg_ref,
                   lmin = 50,
                   lmax = ell[len(ell)-1],
                   cmap = "tab10",
                   bintextpos = [0.15, 0.2],
                   ylim = [0.61, 1.07],
                   legend = param,
                   legendfontsize = 18,
                   linewidth = [2.0],
                   linestyle = ['solid', 'dashed', 'dashdot', 'dotted'],
                   figsize = (18, 13./5),
                   bintextsize = 20,
                   yaxislabelsize = 17,
                   yaxisticklabelsize = 14,
                   xaxisticklabelsize = 20)
# ylim (0.73, 1.07): the band -27% to +7% of w/w_ref - 1
cnu.plot_wtheta_tomo(theta_wtheta = wtheta_theta,
                     theta_wtheta_ref = wtheta_theta_ref,
                     cmap = "tab10",
                     thetashow = [theta_min_arcmin, theta_max_arcmin],
                     bintextpos = [0.85, 0.2],
                     ylim = [0.73, 1.07],
                     legend = param,
                     legendfontsize = 18,
                     linewidth = [2.0],
                     linestyle = ['solid', 'dashed', 'dashdot', 'dotted'],
                     figsize = (18, 13./5),
                     bintextsize = 20,
                     yaxislabelsize = 17,
                     yaxisticklabelsize = 14,
                     xaxisticklabelsize = 20)

## Accuracy settings: how `AccuracyBoost` changes $C_\ell^{gg}$ and $w(\theta)$

The same sweep as for galaxy–galaxy lensing, `AccuracyBoost` from 0.6 to 1.8, for
$C_\ell^{gg}$ with the exact projection below $\ell=150$ and for $w(\theta)$, each
curve drawn relative to `AccuracyBoost = 1`: on a band of ±1.5% for the
spectra and ±10% for $w(\theta)$.

In [ ]:
# 283 multipoles from 10 to 8470, whole numbers for the exact projection
ell = np.arange(10., 8500., 30.)
# AccuracyBoost 0.6 to 1.8; C_gg with the exact projection below ell = 150
param = np.arange(0.6, 2.0, 0.2)

C_gg = []
for x in param:
    C_gg.append(C_gg_tomo(ell = ell, limber = 0, AccuracyBoost = x))
# The AccuracyBoost = 1 spectrum, the reference of the ratios
C_gg_ref = C_gg_tomo(ell = ell, limber = 0)

wtheta_theta = []
for x in param:
    wtheta_theta.append(w_theta(ntheta = 40, AccuracyBoost = x))
wtheta_theta_ref = w_theta(ntheta = 40, AccuracyBoost = 1.0)

In [ ]:
# ylim (0.985, 1.015): the band -1.5% to +1.5% of C/C_ref - 1
cnu.plot_C_gg_tomo(ell = ell,
                   C_gg = C_gg,
                   C_gg_ref = C_gg_ref,
                   lmin = 50,
                   lmax = ell[len(ell)-1],
                   bintextpos = [0.15, 0.2],
                   ylim = [0.985, 1.015],
                   param = param,
                   colorbarlabel = "Accuracy Boost",
                   linewidth = [1.0, 1.3, 1.6, 1.9],
                   linestyle = ['solid', 'dashed', 'dashdot', 'dotted'],
                   figsize = (18, 13./5),
                   bintextsize = 20,
                   yaxislabelsize = 17,
                   yaxisticklabelsize = 14,
                   xaxisticklabelsize = 20)
# ylim (0.9, 1.1): the band -10% to +10% of w/w_ref - 1
cnu.plot_wtheta_tomo(theta_wtheta = wtheta_theta,
                     theta_wtheta_ref = wtheta_theta_ref,
                     thetashow = [theta_min_arcmin, theta_max_arcmin],
                     bintextpos = [0.85, 0.2],
                     ylim = [0.9, 1.1],
                     param = param,
                     colorbarlabel = "Accuracy Boost",
                     linewidth = [1.0, 1.3, 1.6, 1.9],
                     linestyle = ['solid', 'dashed', 'dashdot', 'dotted'],
                     figsize = (18, 13./5),
                     bintextsize = 20,
                     yaxislabelsize = 17,
                     yaxisticklabelsize = 14,
                     xaxisticklabelsize = 20)

## $\chi^2$ against a synthetic DES Y6 data vector

`data/DESY6.dataset` names an all-ones data vector and an identity covariance,
so a $\chi^2$ against them measures nothing. This section installs in `ci` the
three inputs of a survey $\chi^2$, built from the project's own files:

| Input | Content |
| --- | --- |
| Covariance | The `total` array of `covariance/forecast_real.npz`: the Gaussian, super-sample and connected non-Gaussian forecast of EXAMPLE_EVALUATE_COVARIANCE.ipynb, in the 1,300-entry order of the data vector. |
| Mask | `data/DESY6.mask`, the DES Y6 scale cuts: 541 of the 1,300 entries, 152, 58, 272 and 59 of the four blocks. |
| Data vector | Synthetic: this notebook's fiducial prediction with the run settings, not a measurement. |

`init_data_real` reads files, so the first cell writes the covariance as text
into a temporary folder, which Python deletes when the kernel stops. The file
holds the 1,300 variances and the upper triangle of the block the scale cuts
keep: the reader fills each mirror entry itself and zeroes every off-diagonal
entry of a masked row. The cell defines two functions:

- `write_data_vector(datavector, filename)` writes a data vector in the
  two-column layout `init_data_real` reads;
- `initialize_with_data(cov_file, mask_file, data_vector_file)` repeats the
  whole initialization from `initial_setup`, in the order of the likelihood's
  `initialize`, now with the probe selection, the 26 angular bins of the
  dataset and `init_data_real`. The reset is needed because `init_data_real`
  adds the data-vector length to the length CosmoLike holds: installing a
  second data vector without it would double that length.

Here `CLprobe = "3x2pt"` keeps all three probes, so the 541 entries of
the scale cuts enter $\chi^2$.

The second cell defines `get_datavector(...)`, which runs the stages of the
wrappers with every nuisance parameter the likelihood sends and returns
`compute_data_vector_masked()`: all 1,300 entries, masked ones set to zero, with the
shear calibration $(1+m_i)(1+m_j)$ on $\xi_\pm$, $(1+m_j)$ on $\gamma_t$ and the
point-mass term on $\gamma_t$.
`ci.compute_chi2(datavector = t)` then returns
$\chi^2=(d-t)^{\mathsf T}C^{-1}(d-t)$ over the retained entries, with $d$ the
installed data vector, $t$ the prediction and $C^{-1}$ the inverse of the
retained block of the covariance.

The third cell computes the fiducial prediction under the dataset's placeholder
files, whose mask lets `compute_data_vector_masked` run, and installs it as the
synthetic data vector, so its $\chi^2$ is zero by construction. It then prints
the $\chi^2$ of HMcode 2020 without its feedback term
(`halofit_version = "mead2020"`) against it: the size of the YAML's feedback
term in units of the forecast errors, after the scale cuts.

In [ ]:
# tempfile creates the temporary folder of the text files below
import tempfile

# Files of the dataset; ini also names its placeholder data files
ini = IniFile(os.path.normpath(os.path.join(path, data_file)))
scale_cut_file = os.path.join(path, "DESY6.mask")

# The forecast covariance [1300, 1300], in the order of the data vector
forecast_file = os.path.join(os.environ['ROOTDIR'], "projects", "des_y6",
                             "covariance", "forecast_real.npz")
if not os.path.exists(forecast_file):
    raise FileNotFoundError(
        f"{forecast_file} does not exist: the chi^2 sections need the "
        "forecast covariance; run EXAMPLE_EVALUATE_COVARIANCE.ipynb or "
        "covariance/compute_covariance.py first (README, computing covariances)")
# allow_pickle=False reads numbers only; the with block closes the archive
with np.load(forecast_file, allow_pickle = False) as archive:
    forecast_covariance = np.array(archive["total"])

# DESY6.mask: column 0 the entry index 0..1299, column 1 = 1 to keep it
scale_cut_table = np.loadtxt(scale_cut_file)
ndata = scale_cut_table.shape[0]
if forecast_covariance.shape != (ndata, ndata):
    raise ValueError(
        f"forecast covariance shape {forecast_covariance.shape}, scale cuts "
        f"for {ndata} entries: they must agree; recompute the forecast with "
        "EXAMPLE_EVALUATE_COVARIANCE.ipynb for DESY6.dataset")
if not np.array_equal(scale_cut_table[:, 0], np.arange(ndata)):
    raise ValueError(
        f"DESY6.mask does not list the entries 0..{ndata - 1} in order: "
        "restore data/DESY6.mask from the project repository")

# The text layout init_data_real reads: one line "i j C_ij" per stored
# entry. The reader fills the mirror entry (j, i) itself and zeroes every
# off-diagonal entry of a masked row, so the file holds the 1,300
# variances and the upper triangle of the block the scale cuts keep.
cut_entries = np.flatnonzero(scale_cut_table[:, 1] == 1)
masked_entries = np.flatnonzero(scale_cut_table[:, 1] == 0)
# np.triu_indices(n) lists the index pairs (a, b), a <= b, of an n x n block
(block_row, block_column) = np.triu_indices(cut_entries.size)
row_index = np.concatenate([cut_entries[block_row], masked_entries])
column_index = np.concatenate([cut_entries[block_column], masked_entries])
covariance_entries = forecast_covariance[row_index, column_index]

# A folder that lives while the kernel keeps `scratch`; Python deletes it
# when the kernel stops
scratch = tempfile.TemporaryDirectory(prefix = "des_y6_chi2_")
covariance_file = os.path.join(scratch.name, "forecast_total.cov")
# %.17e keeps every digit of a double, so the reader gets the same numbers
np.savetxt(covariance_file,
           np.column_stack([row_index, column_index, covariance_entries]),
           fmt = "%d %d %.17e")


def write_data_vector(datavector, filename):
    """Write a data vector as init_data_real reads it: lines "i d_i".

    Arguments:
      datavector = 1,300 entries in the order of compute_data_vector_masked
      filename   = file written (replaced when it exists)
    """
    entry_index = np.arange(len(datavector))
    np.savetxt(filename, np.column_stack([entry_index, datavector]),
               fmt = "%d %.17e")


def initialize_with_data(cov_file, mask_file, data_vector_file):
    """Run the likelihood's initialization with three data files.

    initial_setup comes first: init_data_real adds the data-vector length
    to the length CosmoLike holds, so a second call without the reset
    would double it. The sequence is the one of the initialization cell,
    plus the probe selection, the dataset binning and init_data_real, in
    the order of the likelihood's initialize.

    Arguments:
      cov_file, mask_file, data_vector_file = text files in the layouts
                 init_data_real reads, 1,300 entries each

    Side effects:
      replaces every CosmoLike setting; ci.compute_chi2, ci.get_mask and
      ci.get_inv_cov_masked then use the three files.
    """
    ci.initial_setup()
    ci.set_omp_threads(n = threads)
    ci.init_probes(possible_probes = CLprobe)
    ci.init_binning(ntheta_bins = int(ini.int("n_theta")),
                    theta_min_arcmin = ini.float("theta_min_arcmin"),
                    theta_max_arcmin = ini.float("theta_max_arcmin"))
    ci.init_photoz_conventions(
        interpolation_type = int(photoz_interpolation_type),
        zmid_convention = int(photoz_zmid_convention))
    # initial_setup resets the FAST-PT grid fraction to the C default; set
    # it again before the first init_accuracy_boost
    ci.init_fpt_internal_boost(internal_boost = float(internal_accuracyboost))
    ci.init_nonlimber_accuracy_boost(nonlimber_boost = float(nonlimber_accuracyboost))
    ci.init_adopt_limber_gs(adopt_limber_gs = int(adopt_limber_gs))
    ci.init_adopt_limber_gg(adopt_limber_gg = int(adopt_limber_gg))
    ci.init_include_HOD_GX(include_HOD_GX = 0)
    ci.init_include_halo_IA(include_halo_IA = 0)
    ci.init_ntable_lmax(lmax = int(lmax))
    ci.init_accuracy_boost(accuracy_boost = 1.0,
                           integration_accuracy = int(integration_accuracy))
    ci.init_cosmo_runmode(is_linear = False)
    ci.init_redshift_distributions_from_files(
          lens_multihisto_file = ini.relativeFileName('nz_lens_file'),
          lens_ntomo = int(ini.int("lens_ntomo")),
          source_multihisto_file = ini.relativeFileName('nz_source_file'),
          source_ntomo = int(ini.int("source_ntomo")))
    ci.init_data_real(COV = cov_file, MASK = mask_file, DATA = data_vector_file)
    ci.init_IA(ia_model = int(IA_model),
               ia_redshift_evolution = int(IA_redshift_evolution),
               ia_code = int(IA_code))
    # The bias codes of the wrappers; a shear-only selection has no
    # lens bias
    if CLprobe != "xi":
        ci.init_bias(bias_model = bias_model)

In [ ]:
def get_datavector(omegam = omegam,
                   omegab = omegab,
                   H0 = H0,
                   ns = ns,
                   As_1e9 = As_1e9,
                   w = w,
                   w0pwa = w0pwa,
                   A1 = [DES_A1_1, DES_A1_2, 0, 0],
                   A2 = [DES_A2_1, DES_A2_2, 0, 0],
                   BTA = [DES_BTA_1, 0, 0, 0],
                   shear_photoz_bias = [DES_DZ_S1, DES_DZ_S2, DES_DZ_S3, DES_DZ_S4],
                   M = [DES_M1, DES_M2, DES_M3, DES_M4],
                   lens_photoz_bias = [DES_DZ_L1, DES_DZ_L2, DES_DZ_L3,
                                       DES_DZ_L4, DES_DZ_L5, DES_DZ_L6],
                   galaxy_bias_b1 = [DES_B1_1, DES_B1_2, DES_B1_3,
                                     DES_B1_4, DES_B1_5, DES_B1_6],
                   galaxy_bias_b2 = [0, 0, 0, 0, 0, 0],
                   galaxy_bias_bmag = [DES_BMAG_1, DES_BMAG_2, DES_BMAG_3,
                                       DES_BMAG_4, DES_BMAG_5, DES_BMAG_6],
                   PM = [DES_PM1, DES_PM2, DES_PM3, DES_PM4, DES_PM5, DES_PM6],
                   halofit_version = halofit_version,
                   AccuracyBoost = 1.0,
                   kmax = kmax,
                   k_per_logint = k_per_logint,
                   CAMBAccuracyBoost = CAMBAccuracyBoost,
                   CLAccuracyBoost = 1.0,
                   CLIntegrationAccuracy = integration_accuracy):
    """Return the masked model vector [1300] at one parameter point.

    Runs the stages of the wrappers with every nuisance parameter
    the likelihood sends, then compute_data_vector_masked: masked entries
    are zero, and the shear calibration and point-mass terms are applied.
    The binning is reset to the dataset's 26 bins, which the real-space
    wrappers may have changed.
    """
    # Stage 1, CAMB
    (log10k_interp_2D, z_interp_2D, lnPL, lnPNL, G_growth, z_growth,
     z_interp_1D, chi, omegan2, lnPL_cb) = cnu.get_camb_cosmology(
        omegam = omegam, omegab = omegab, H0 = H0, ns = ns,
        As_1e9 = As_1e9, w = w, w0pwa = w0pwa, mnu = mnu,
        non_linear_emul = non_linear_emul,
        halofit_version = halofit_version,
        AccuracyBoost = AccuracyBoost, kmax = kmax,
        k_per_logint = k_per_logint,
        CAMBAccuracyBoost = CAMBAccuracyBoost,
        CLAccuracyBoost = CLAccuracyBoost)

    # Stage 2, accuracy of the C code, as in the wrappers
    ci.set_omp_threads(n = threads)
    CLAccuracyBoost = CLAccuracyBoost * AccuracyBoost
    CLIntegrationAccuracy = max(0, CLIntegrationAccuracy + abs(3*(CLAccuracyBoost-1.0)))
    ci.init_ntable_lmax(lmax = int(lmax + 20000*(CLAccuracyBoost-1)))
    ci.init_accuracy_boost(accuracy_boost = CLAccuracyBoost,
                           integration_accuracy = int(CLIntegrationAccuracy))
    ci.init_photoz_conventions(
        interpolation_type = int(photoz_interpolation_type),
        zmid_convention = int(photoz_zmid_convention))
    ci.init_binning(ntheta_bins = int(ntheta),
                    theta_min_arcmin = theta_min_arcmin,
                    theta_max_arcmin = theta_max_arcmin)

    # Stage 3: copy the CAMB tables into CosmoLike
    ci.set_cosmology(omegam = omegam,
                     omegab = omegab,
                     H0 = H0,
                     log10k_2D = log10k_interp_2D,
                     z_2D = z_interp_2D,
                     lnP_linear = lnPL,
                     lnP_linear_cb = lnPL_cb,
                     lnP_nonlinear = lnPNL,
                     G = G_growth,
                     z_G = z_growth,
                     z_1D = z_interp_1D,
                     chi = chi,
                     omegan2 = omegan2)

    # Stage 4: every nuisance parameter, as the likelihood's
    # set_lens_related and set_source_related send them
    ci.set_point_mass(PMV = PM)
    ci.set_nuisance_bias(B1 = galaxy_bias_b1,
                         B2 = galaxy_bias_b2,
                         B_MAG = galaxy_bias_bmag,
                         B3nl = [0, 0, 0, 0, 0, 0],
                         BK = [0, 0, 0, 0, 0, 0])
    ci.set_nuisance_clustering_photoz(bias = lens_photoz_bias)
    ci.set_nuisance_shear_calib(M = M)
    ci.set_nuisance_shear_photoz(bias = shear_photoz_bias)
    ci.set_nuisance_ia(A1 = A1, A2 = A2, B_TA = BTA)
    # No tabulated feedback in the data vector
    ci.reset_bary_struct()

    # Stage 5: all 1,300 entries, masked ones set to zero
    return np.array(ci.compute_data_vector_masked())

In [ ]:
# The dataset's placeholder files (identity covariance, all-ones data,
# ones.mask) only let compute_data_vector_masked run for the first
# prediction: the synthetic data vector
initialize_with_data(cov_file = ini.relativeFileName('cov_file'),
                     mask_file = ini.relativeFileName('mask_file'),
                     data_vector_file = ini.relativeFileName('data_file'))
dv_fiducial = get_datavector()

# The forecast covariance, the scale cuts and the fiducial prediction as
# the synthetic data vector
fiducial_data_file = os.path.join(scratch.name, "fiducial.modelvector")
write_data_vector(datavector = dv_fiducial, filename = fiducial_data_file)
initialize_with_data(cov_file = covariance_file,
                     mask_file = scale_cut_file,
                     data_vector_file = fiducial_data_file)

# get_mask returns the 0/1 flag of every data-vector entry: their sum is
# the number of entries that enter chi^2
n_retained = int(np.sum(ci.get_mask()))
chi2_fiducial = ci.compute_chi2(datavector = get_datavector())
print("chi2 of the fiducial prediction (halofit_version = %s) = %.4f "
      "over %d data points" % (halofit_version, chi2_fiducial, n_retained))
# HMcode 2020 without its feedback term, against the same data vector
dv_no_feedback = get_datavector(halofit_version = "mead2020")
chi2_no_feedback = ci.compute_chi2(datavector = dv_no_feedback)
print("chi2 of HMcode 2020 without feedback (mead2020) = %.4f "
      "over %d data points" % (chi2_no_feedback, n_retained))

## Quadrature check: $\Delta\chi^2$ against the integration level

The next cell evaluates $\chi^2$ at quadrature levels 0 to 9, everything else
fixed, and subtracts the level-0 value; the plot after it shows
$\Delta\chi^2=\chi^2(\text{level})-\chi^2(0)$ against the level. Level 0 is the
setting of the synthetic data vector, so $\chi^2(0)=0$ and each $\Delta\chi^2$ is
the $\chi^2$ of the shift between that level's prediction and level 0. Each level
selects a fixed Gauss–Legendre rule, a set of points and weights for the
line-of-sight integrals: from 64 to 128 points at level 0,
depending on the probe, up to 1024. From level 4 on every probe uses its
1024-point rule, so the curve is flat there.

The frozen-reference tests of the Cocoa projects that ship a survey covariance
accept $\Delta\chi^2<0.2$. The DES Y6 tests compare data-vector entries instead,
because the shipped covariance is the identity: run
`python -m pytest ./projects/des_y6/tests/data_vector` from `cocoa/Cocoa` for
the accuracy, cache-consistency and frozen-reference checks.

In [ ]:
param = np.arange(0, 10, 1)
chi2 = []
for x in param:
    datavector = get_datavector(CLIntegrationAccuracy = x)
    chi2.append(ci.compute_chi2(datavector = datavector))
chi2 = np.array(chi2)
# Delta chi^2 relative to quadrature level 0, the setting of the synthetic
# data vector (chi2[0] = 0)
chi2 = chi2 - chi2[0]

In [ ]:
# Delta chi^2 of each quadrature level against level 0
plt.plot(param, chi2, marker = 'o')
plt.xlabel("quadrature level (CLIntegrationAccuracy)")
plt.ylabel(r"$\Delta\chi^2$ against level 0")
plt.show()

## Interpolation check: $\Delta\chi^2$ against `CLAccuracyBoost`

The next cell evaluates $\chi^2$ at `CLAccuracyBoost` = 1, 2, ..., 9 and subtracts
the boost-1 value, zero for the same reason. This boost multiplies the sizes of
CosmoLike's interpolation tables and the redshift and wavenumber grids of the
CAMB tables, and raises the multipole cutoff of `init_ntable_lmax` by 20,000 per
unit; CAMB's own accuracy stays fixed. `get_datavector` also raises the
quadrature level by $3(\text{boost}-1)$ above the default level 0, so this scan
refines interpolation and quadrature together; from boost 3 on, the quadrature
uses its largest rule. The highest boosts build the largest tables and take the
longest.

In [ ]:
param = np.arange(1, 10, 1)
chi2 = []
for x in param:
    datavector = get_datavector(CLAccuracyBoost = x)
    chi2.append(ci.compute_chi2(datavector = datavector))
chi2 = np.array(chi2)
# Delta chi^2 relative to CLAccuracyBoost = 1, the setting of the synthetic
# data vector (chi2[0] = 0); the quadrature level is 3 (boost - 1), so it
# rises with the boost
chi2 = chi2 - chi2[0]

In [ ]:
# Delta chi^2 of each CLAccuracyBoost against boost 1
plt.plot(param, chi2, marker = 'o')
plt.xlabel("CLAccuracyBoost")
plt.ylabel(r"$\Delta\chi^2$ against boost 1")
plt.show()

## Baryonic feedback from the `bfmt` theory block

The two sections *Baryonic feedback from tabulated simulations in $C_\ell^{gs}$
and $\gamma_t(\theta)$* and *... in $C_\ell^{gg}$ and $w(\theta)$* apply fixed
simulation ratios: `ci.init_baryons_contamination` multiplies CosmoLike's
nonlinear matter power by the power spectrum of one hydrodynamical simulation
divided by that of its dark-matter-only counterpart, as tabulated in the HDF5 file
`baryons_logPkR.h5`. This section takes the other path: the `bfmt` theory block
(`external_modules/code/baryon_suppression`) evaluates parametric models of the
suppression $S(k,z)=P_{\rm feedback}(k,z)/P_{\rm DMO}(k,z)$, the matter power
with baryonic feedback over the dark-matter-only (DMO) power, at any value of
their parameters; $S<1$ means suppressed power. The block implements seven
models:

- three SP(k) relations (Salcido et al. 2023), which map the baryon fraction
  retained in halos, as a function of halo mass and redshift, to $S(k,z)$; they
  differ in how that fraction depends on mass;
- four emulators, fast interpolators trained on suites of simulations and valid
  inside the parameter box they were trained on: BCEmu (a baryonification
  model), the FLAMINGO response emulator, BACCOemu and BCemu2025.

The likelihoods expose both paths: `add_baryons_on_dv` with
`which_bsims_add_on_dv` applies one tabulated simulation, as the two sections
above do, and `external_baryon_suppression: true` requests $S(k,z)$ from `bfmt`
on the likelihood's $(z,k)$ grid and adds $\ln S$ to the nonlinear $\ln P$ table
before `set_cosmology`; `EXAMPLE_EVALUATE1_BARYONS.yaml` runs BCEmu this way.
Instead of re-implementing the models, this section drives the block itself
through a minimal Cobaya model. Cobaya is the framework in which Cocoa's
likelihoods run, and a *theory block* is a Cobaya component that computes a
product other components request; the model holds CAMB, `bfmt`, and the `one`
likelihood, which always returns zero and only makes Cobaya run the theory
blocks. $S$ multiplies the gravity-only Takahashi spectrum, as in the tabulated
sections: on HMcode's feedback term, the run-settings model, it would count
feedback twice.

The cells below

1. list the six methods run here, each with fixed parameter values (BACCOemu is
   left out, for the reason its comment gives);
2. define `get_baryon_suppression`, which returns $S(k,z)$, and `compute_probes`,
   which returns $C_\ell^{gs}$, $\gamma_t(\theta)$, the masked 3×2pt data vector and
   its $\chi^2$, with an optional $S$ applied;
3. compute the prediction without feedback, install it as the synthetic data
   vector of the $\chi^2$ section's machinery, compute the prediction with each
   method, and print the $\chi^2$ table;
4. draw $C_\ell^{gs}$ and $\gamma_t(\theta)$ with each method relative to no
   feedback.

**Before running.** The section needs `bfmt` and its emulators (see
[baryonic feedback](README.md#baryons) in the README) and the functions of the
$\chi^2$ section above. It runs CAMB 13 times, once for the prediction without
feedback and twice per method (in the Cobaya model and in `compute_probes`), a
few minutes in total.

In [ ]:
# One entry per feedback method: (label, bfmt options, fixed parameter
# point). The SP(k) points are pyspk's documented examples; the emulator
# points are the fiducial values quoted in the example yamls. BACCOemu
# is commented out so that the table lists the same six methods as the
# other projects' notebooks; it would run here: its omega_baryon
# training box starts at 0.04001, below this project's omegab = 0.049.
BARYON_METHODS = [
    ("SP(k) power law", {"baryon_model": 1, "spk_fb_model": 1},
     {"fb_a_spk": 0.4, "fb_pow_spk": 0.3}),
    ("SP(k) Akino et al. 2022", {"baryon_model": 1, "spk_fb_model": 2},
     {"alpha_spk": 4.189, "beta_spk": 1.273, "gamma_spk": 0.298}),
    # this point matches the Akino relation's amplitude at the pivot
    # and stays inside SP(k)'s calibrated baryon-fraction band over
    # the full z grid; pyspk's documented example (0.3, 1.1, 0.2,
    # 0.5) exits the band at z >~ 1.4, and the block then falls back
    # to unity for those redshifts
    ("SP(k) double power law", {"baryon_model": 1, "spk_fb_model": 3},
     {"epsilon_spk": 0.66, "alpha_spk": 0.35, "beta_spk": 0.2,
      "gamma_spk": 0.3}),
    ("BCEmu", {"baryon_model": 2},
     {"log10Mc_bcemu": 13.32, "mu_bcemu": 0.93, "thej_bcemu": 4.235,
      "gamma_bcemu": 2.25, "delta_bcemu": 6.40, "eta_bcemu": 0.15,
      "deta_bcemu": 0.14}),
    ("Flamingo", {"baryon_model": 3},
     {"fgas_sigma_flamingo": 0.0, "mstar_sigma_flamingo": 0.0,
      "jet_frac_flamingo": 0.0}),
    #("BACCOemu", {"baryon_model": 4},
    # {"M_c_baccoemu": 14.0, "eta_baccoemu": -0.3, "beta_baccoemu": -0.22,
    #  "M1_z0_cen_baccoemu": 10.5, "theta_inn_baccoemu": -0.86}),
    ("BCemu2025", {"baryon_model": 5},
     {"Theta_co_bcemu25": 0.3, "log10Mc_bcemu25": 13.1, "mu_bcemu25": 1.0,
      "delta_bcemu25": 6.0, "eta_bcemu25": 0.10, "deta_bcemu25": 0.22,
      "Nstar_bcemu25": 0.028}),
]

**The two functions.**

- `get_baryon_suppression(theory_options, point, z_grid, log10k_grid)` builds the
  minimal Cobaya model, evaluates it at the notebook's fiducial cosmology plus the
  method's parameters, and returns a dictionary `{z: S}` with one array over the
  requested wavenumbers per redshift. Each call runs CAMB once.
- `compute_probes(sup=None)` runs the five stages of the wrappers at the fiducial
  point, on the Takahashi spectrum. Between CAMB and `set_cosmology`, when `sup` is
  given, it adds $\ln S(k,z)$ to a copy of the nonlinear $\ln P$ table, the
  operation the likelihoods apply. It returns a dictionary with $C_\ell^{gs}$ at the
  multipoles `ell`, $\gamma_t$ at the angles `theta` (arcmin) of the 26 data-vector
  bins, the masked data vector `dv`, its `chi2`, and the grids `z_grid` and
  `log10k_grid` of the CAMB tables.

Both read the fiducial values from the notebook at every call and take no
cosmology arguments, so a suppression and the prediction it multiplies always
share one cosmology. `compute_probes` uses the CAMB accuracy of the run settings
(accuracy 1.05, $k_{\rm max}=5$/Mpc, 10 samples per logarithmic interval before
the helper's scaling), CosmoLike tables at boost 1 and quadrature level 0, the
level of `EXAMPLE_EVALUATE1.yaml`. They stay fixed because another boost changes
the $(z,k)$ grid, and $S$ must be given on the grid it multiplies.

**Units at the handoff.** `cnu.get_camb_cosmology` returns $\log_{10}k$ with $k$ in
$h$/Mpc, the unit of `set_cosmology`, while `get_baryon_suppression`, like the
likelihoods' request to `bfmt`, reads $k$ in 1/Mpc. The `log10k_grid` that
`compute_probes` returns is therefore $\log_{10}k+\log_{10}h$, with $h=H_0/100$ for
$H_0$ in km/s/Mpc: the same nodes, in 1/Mpc. The $h$/Mpc grid handed over
unchanged would evaluate $S$ at $k/h$, about $1.49\,k$ at $h=0.67$, instead of
at $k$.

In [ ]:
def get_baryon_suppression(theory_options, point, z_grid, log10k_grid):
    """Return the suppression S(k, z) of the bfmt theory block on a grid.

    S = P(k) with baryonic feedback / P(k) without it. Builds a minimal
    Cobaya model (CAMB + bfmt + the likelihood "one", which returns
    ln L = 0 and only makes the model complete), requests the
    baryon_suppression product at the given grid (k in 1/Mpc, as the
    Cosmolike likelihoods send it; the block converts to h/Mpc
    internally), evaluates it at the notebook's fiducial cosmology (the
    names of the "Fiducial parameters" cell, read at every call), and
    returns {z: S array over k}.

    Arguments:
      theory_options = bfmt options dict, e.g. {"baryon_model": 2}.
      point   = {parameter name: value} for the method's feedback
                parameters, fixed in the model.
      z_grid  = redshifts of the evaluation grid.
      log10k_grid = log10 of the wavenumbers, read as k in 1/Mpc.

    Returns:
      {z: 1D S array over k}, one entry per z_grid value.
    """
    from cobaya.model import get_model
    # Cobaya model description: CAMB at the notebook's fiducial
    # cosmology (tau = 0.0543, a Planck 2018 value, only completes
    # CAMB's input; omch2 subtracts the massive-neutrino density),
    # bfmt with the caller's options and feedback parameters, and
    # debug = 50 (logging.CRITICAL: only critical messages print)
    info = {
        "likelihood": {"one": None},
        "theory": {
            # no "path" for camb: the session already imported it,
            # and cobaya accepts the loaded module as is
            "camb": {"extra_args": {"halofit_version": "takahashi",
                                    "dark_energy_model": "ppf"}},
            "bfmt": dict({"python_path": os.environ["ROOTDIR"]
                          + "/external_modules/code/baryon_suppression"},
                         **theory_options),
        },
        "params": dict({
            "As": {"value": As_1e9*1e-9},
            "ns": ns, "H0": H0, "mnu": mnu, "tau": 0.0543,
            "w": w,
            "ombh2": omegab*(H0/100)**2,
            "omch2": (omegam-omegab)*(H0/100)**2
                     - (mnu*(3.046/3)**0.75)/94.0708,
            "omegam": {"derived": True, "latex": r"\Omega_m"},
        }, **point),
        "debug": 50,
    }
    model = get_model(info)
    model.add_requirements({"baryon_suppression": {
        "z": z_grid, "k": np.power(10.0, log10k_grid)}})
    # every parameter is fixed, so the point to evaluate is the empty
    # dictionary; the call runs CAMB and bfmt once
    model.logposterior({})
    return model.provider.get_baryon_suppression()

In [ ]:
def compute_probes(sup = None, ell = None):
    """Compute C_ell^gs, gamma_t, the masked 3x2pt data vector and its chi^2.

    Runs the five stages of the wrappers at the fiducial point of the
    "Fiducial parameters" cell, whose names are read from the notebook at
    every call, as get_baryon_suppression reads them: a suppression and the
    prediction it multiplies share one cosmology. CAMB runs the
    gravity-only Takahashi Halofit, the spectrum S multiplies. The run
    settings and ini (the dataset file, which fixes the data-vector
    binning) are read from the notebook as well. sup = None computes the
    prediction without feedback; otherwise sup is the {z: S array}
    dictionary of get_baryon_suppression, applied the way the likelihoods
    apply it: ln P_nonlinear(k, z_i) += ln S(k, z_i) at every redshift node
    z_i. The accuracy settings stay fixed because a different boost
    changes the (z, k) grid on which sup must be given.

    Arguments:
      sup = suppression dictionary on the CAMB interpolation grid (one
            entry per z of z_grid, each an array over the k of
            log10k_grid), or None.
      ell = multipoles of the returned C_ell^gs, or None for
            np.arange(25, 3000, 15), the multipole range of the figure.

    Returns:
      dict with
        ell         = the multipoles, [n_ell];
        C_gs        = Limber C_ell^gs, [n_ell, n_lens, n_source], plain
                      dimensionless spectra;
        theta       = area-weighted centers of the data-vector angular
                      bins, [n_theta], in arcmin;
        gammat      = gamma_t, [n_theta, n_lens, n_source];
        dv          = the masked data vector: all 1,300 entries, masked
                      ones set to zero;
        chi2        = (d - t)^T C^-1 (d - t) over the retained entries,
                      with d the installed synthetic data vector;
        z_grid      = the redshift nodes of the CAMB tables;
        log10k_grid = log10 of the wavenumber nodes, k in 1/Mpc, the unit
                      get_baryon_suppression takes.

    Raises:
      RuntimeError when no data vector is installed: compute_chi2 would
      otherwise stop the whole Python process.
    """
    # the chi^2 section installs the mask; without it the mask is empty,
    # and the chi^2 call below would end the kernel, not raise
    n_mask = len(ci.get_mask())
    if n_mask == 0:
        raise RuntimeError(
            "compute_probes: CosmoLike holds no data vector (its mask has "
            "0 entries); the chi^2 needs the forecast covariance, the scale "
            "cuts and a data vector: run the chi^2 section first")

    if ell is None:
        ell = np.arange(25., 3000., 15.)

    # Stage 1, CAMB on the gravity-only Takahashi spectrum, with the CAMB
    # accuracy of the run settings
    (log10k_interp_2D, z_interp_2D, lnPL, lnPNL, G_growth, z_growth,
     z_interp_1D, chi, omegan2, lnPL_cb) = cnu.get_camb_cosmology(
        omegam = omegam, omegab = omegab, H0 = H0, ns = ns,
        As_1e9 = As_1e9, w = w, w0pwa = w0pwa, mnu = mnu,
        non_linear_emul = non_linear_emul, halofit_version = "takahashi",
        AccuracyBoost = 1.0, kmax = kmax, k_per_logint = k_per_logint,
        CAMBAccuracyBoost = CAMBAccuracyBoost, CLAccuracyBoost = 1.0)

    # a private copy of ln P_nonlinear, modified in place below
    lnPNL = np.array(lnPNL, copy = True)
    if sup is not None:
        for i, z_val in enumerate(z_interp_2D):
            # every k of redshift node i sits at stride len(z_interp_2D)
            # in the flattened (Fortran-order) table, the layout
            # set_cosmology expects. sup[z_val] looks the redshift up by
            # exact float equality, which holds because sup was computed
            # on this same z grid (the z_grid returned below).
            lnPNL[i :: len(z_interp_2D)] += np.log(sup[z_val])

    # Stage 2, accuracy of the C code: tables at boost 1, whose largest
    # tabulated multipole is lmax, and the quadrature level of the run
    # settings
    ci.set_omp_threads(n = threads)
    ci.init_ntable_lmax(lmax = int(lmax))
    ci.init_photoz_conventions(
        interpolation_type = int(photoz_interpolation_type),
        zmid_convention = int(photoz_zmid_convention))
    # init_fpt_internal_boost comes first, as in the likelihood, so the
    # FAST-PT grid fraction is the configured one even when this is the
    # first init_accuracy_boost of the process
    ci.init_fpt_internal_boost(internal_boost = float(internal_accuracyboost))
    ci.init_accuracy_boost(accuracy_boost = 1.0,
                           integration_accuracy = int(integration_accuracy))
    # the binning of the installed data vector: the gamma_t and w_theta
    # wrappers may have left another one
    ci.init_binning(ntheta_bins = int(ini.int("n_theta")),
                    theta_min_arcmin = ini.float("theta_min_arcmin"),
                    theta_max_arcmin = ini.float("theta_max_arcmin"))

    # Stage 3: copy the CAMB tables, with S applied, into CosmoLike
    ci.set_cosmology(omegam = omegam,
                     omegab = omegab,
                     H0 = H0,
                     log10k_2D = log10k_interp_2D,
                     z_2D = z_interp_2D,
                     lnP_linear = lnPL,
                     lnP_linear_cb = lnPL_cb,
                     lnP_nonlinear = lnPNL,
                     G = G_growth,
                     z_G = z_growth,
                     z_1D = z_interp_1D,
                     chi = chi,
                     omegan2 = omegan2)

    # Stage 4, nuisance parameters at the fiducial point: source side (the
    # IA setter reads A1[0:2], A2[0:2] and B_TA[0]; zeros pad the lists),
    # lens side, then the point masses of gamma_t
    ci.set_nuisance_shear_calib(M = [DES_M1, DES_M2, DES_M3, DES_M4])
    ci.set_nuisance_shear_photoz(
        bias = [DES_DZ_S1, DES_DZ_S2, DES_DZ_S3, DES_DZ_S4])
    ci.set_nuisance_ia(A1 = [DES_A1_1, DES_A1_2, 0, 0],
                       A2 = [DES_A2_1, DES_A2_2, 0, 0],
                       B_TA = [DES_BTA_1, 0, 0, 0])
    ci.set_nuisance_clustering_photoz(
        bias = [DES_DZ_L1, DES_DZ_L2, DES_DZ_L3, DES_DZ_L4, DES_DZ_L5, DES_DZ_L6])
    ci.set_nuisance_bias(
        B1 = [DES_B1_1, DES_B1_2, DES_B1_3, DES_B1_4, DES_B1_5, DES_B1_6],
        B2 = [0, 0, 0, 0, 0, 0],
        B_MAG = [DES_BMAG_1, DES_BMAG_2, DES_BMAG_3,
                 DES_BMAG_4, DES_BMAG_5, DES_BMAG_6],
        B3nl = [0, 0, 0, 0, 0, 0],
        BK = [0, 0, 0, 0, 0, 0])
    ci.set_point_mass(PMV = [DES_PM1, DES_PM2, DES_PM3, DES_PM4, DES_PM5, DES_PM6])
    # S, when given, already sits in lnPNL: the tabulated simulation ratio
    # of init_baryons_contamination stays off
    ci.reset_bary_struct()

    # Stage 5: the spectra, the masked data vector and its chi^2, then
    # gamma_t at the data-vector binning
    C_gs = ci.C_gs_tomo_limber(l = ell)
    dv = np.array(ci.compute_data_vector_masked())
    chi2 = ci.compute_chi2(datavector = dv)
    theta = ci.get_binning_real_space()
    gammat = ci.w_gammat_tomo()
    # get_baryon_suppression takes k in 1/Mpc, but the CAMB helper
    # returns this grid in h/Mpc: convert here, at the one place
    # that links the two, so S(k) is evaluated at the physical k.
    log10k_grid = log10k_interp_2D + np.log10(H0/100.0)
    return {"ell": ell,
            "C_gs": C_gs,
            "theta": theta,
            "gammat": gammat,
            "dv": dv,
            "chi2": chi2,
            "z_grid": z_interp_2D,
            "log10k_grid": log10k_grid}

**Without and with feedback.** The first cell computes the prediction without
feedback and installs it, with the forecast covariance and the DES Y6 scale cuts,
as the synthetic data vector of this table. It prints that prediction's $\chi^2$,
zero by construction, the number of entries that enter $\chi^2$ (541) and the
length of the masked data vector (1,300: the masked entries are zero).

The second cell loops over the six methods: for each it computes $S(k,z)$ on the
$(z,k)$ grid of the CAMB tables, then the prediction with that suppression, and
prints its $\chi^2$. The third cell prints the table, one row per prediction, no
feedback first:

- $\chi^2$ against the synthetic data vector, over the 541 retained entries;
- $\Delta\chi^2=\chi^2-\chi^2_{\rm none}$, the change from the prediction without
  feedback;
- the $\chi^2$ of the shift, $(t-t_{\rm none})^{\mathsf T}C^{-1}(t-t_{\rm none})$,
  with $t$ the masked prediction and $C$ the forecast covariance: the distance
  between the predictions with and without feedback in units of the forecast
  errors, independent of the data vector.

The data vector here is synthetic and equal to $t_{\rm none}$, so the three
columns coincide up to rounding; the table keeps them, as in the notebooks of
the projects with measured data vectors, where they differ. A value well above
one means that the DES Y6 scale cuts do not protect a 3×2pt analysis from
feedback of that strength.

In [ ]:
# ref holds ell, C_gs, theta, gammat, the masked data vector dv, its chi2,
# and the z and log10 k grids of the CAMB tables
ref = compute_probes()

# The prediction without feedback becomes the synthetic data vector of the
# table, with the forecast covariance and the scale cuts of the chi^2
# section; its chi^2 against itself is then zero
no_feedback_data_file = os.path.join(scratch.name, "no_feedback.modelvector")
write_data_vector(datavector = ref["dv"], filename = no_feedback_data_file)
initialize_with_data(cov_file = covariance_file,
                     mask_file = scale_cut_file,
                     data_vector_file = no_feedback_data_file)
ref["chi2"] = ci.compute_chi2(datavector = ref["dv"])

# get_mask returns the 0/1 flag of every data-vector entry: their sum is
# the number of entries that enter chi^2
n_retained = int(np.sum(ci.get_mask()))
print("no feedback: chi2 = %.4f over %d data points, data vector length = %d" %
      (ref["chi2"], n_retained, len(ref["dv"])))

In [ ]:
# For each method: one Cobaya model (CAMB + bfmt) gives S(k, z) on the grid
# of ref, then one data vector with S applied to the nonlinear power.
# ref["log10k_grid"] is already in 1/Mpc, the unit get_baryon_suppression
# takes.
results = {}
for label, theory_options, point in BARYON_METHODS:
    sup = get_baryon_suppression(theory_options = theory_options,
                                 point = point,
                                 z_grid = ref["z_grid"],
                                 log10k_grid = ref["log10k_grid"])
    results[label] = compute_probes(sup = sup)
    # flush = True prints each line as soon as its method is done
    print("%-28s chi2 = %10.4f" % (label, results[label]["chi2"]),
          flush = True)

In [ ]:
# C^-1 as CosmoLike holds it: a 1300 x 1300 matrix whose masked rows and
# columns are zero, the matrix compute_chi2 contracts, so the difference of
# two masked data vectors can be contracted with it directly
inv_cov = np.array(ci.get_inv_cov_masked())

# the label of every (label, bfmt options, point) entry, in table order
labels = []
for label, theory_options, point in BARYON_METHODS:
    labels.append(label)

print("chi2 over %d data points (the DES Y6 scale cuts; synthetic data vector "
      "= the prediction without feedback)" % n_retained)
print("%-28s %12s %12s %14s" % ("method", "chi2", "Delta chi2", "chi2 of shift"))
# the no-feedback row first, then one row per method
rows = [("no feedback", ref)]
for label in labels:
    rows.append((label, results[label]))
for label, result in rows:
    delta_chi2 = result["chi2"] - ref["chi2"]
    # (t - t_none)^T C^-1 (t - t_none); @ is the matrix product
    shift = result["dv"] - ref["dv"]
    chi2_shift = shift @ inv_cov @ shift
    print("%-28s %12.4f %12.4f %14.4f" %
          (label, result["chi2"], delta_chi2, chi2_shift))

**$C_\ell^{gs}$ with feedback.** One row per source bin and one column per lens
bin, labeled (lens, source), with
$C_\ell^{gs,\,\rm feedback}/C_\ell^{gs,\,\rm none}-1$ for $25\le\ell<3000$, one color
per method: the colorbar values only order the methods, the legend above the
panels names them, and the two lightest colors are drawn dashed. The first cell
defines `fitted_ratio_band`, which fits the band to the curves: from the lowest to
the highest drawn value, zero included, plus 10% of that width at each end.
Expect no change at low $\ell$ and a suppression that grows with $\ell$. Pairs whose
source bin lies in front of the lens bin carry little lensing signal, so their
ratios say little about feedback.

In [ ]:
def fitted_ratio_band(curves, reference):
    """Return a band (lower, upper) fitted to every curve/reference - 1.

    The band spans the smallest and the largest value of
    curve/reference - 1 over all curves and panels, includes 0 (the
    no-feedback line), and adds 10% of its width at each end, so the
    curves fill the panels without touching the frame. Panels whose
    reference is identically zero (pairs excluded from the data vector)
    are skipped, as the plotters skip them.

    Arguments:
      curves    = list of arrays [n_x, n_lens, n_source], one per method;
                  n_x counts multipoles or angular bins.
      reference = the no-feedback array, with the same shape.

    Returns:
      (1 + low - padding, 1 + high + padding): the band written around 1,
      the ylim convention of the cnu plotters with a reference.
    """
    low = 0.0
    high = 0.0
    n_lens = reference.shape[1]
    n_source = reference.shape[2]
    for curve in curves:
        for i in range(n_lens):
            for j in range(n_source):
                if not np.any(reference[:, i, j]):
                    continue
                ratio = curve[:, i, j]/reference[:, i, j] - 1
                low = min(low, np.min(ratio))
                high = max(high, np.max(ratio))
    padding = 0.1*(high - low)
    return (1 + low - padding, 1 + high + padding)

In [ ]:
# param only orders the curve colors, one value per method; the legend
# above the panels names the methods
param = np.arange(1.0, 1.0 + len(labels))

# One linestyle per method, in BARYON_METHODS order, so every panel draws
# a method with the same style: the colormap gives the second method a
# light yellow and the fourth a light cyan, which get a long dash
# ((0, (8, 3)): 8 points drawn, 3 skipped); a wide line (linewidth, one
# entry repeated for every curve) keeps all six curves visible
method_linestyles = ["solid", (0, (8, 3)), "solid", (0, (8, 3)), "solid", "solid"]
if len(method_linestyles) != len(labels):
    raise ValueError(
        "method_linestyles has %d entries for %d methods in BARYON_METHODS: "
        "give one linestyle per method" % (len(method_linestyles), len(labels)))

C_gs_list = []
for label in labels:
    C_gs_list.append(results[label]["C_gs"])

C_gs_band = fitted_ratio_band(curves = C_gs_list, reference = ref["C_gs"])

# bintextpos (0.15, 0.2): the lower-left corner of each panel, which the
# curves leave free (they sit at zero at low ell)
cnu.plot_C_gs_tomo_limber(ell = ref["ell"],
                          C_gs = C_gs_list,
                          C_gs_ref = ref["C_gs"],
                          param = param,
                          colorbarlabel = "bfmt method",
                          lmin = 25,
                          lmax = 3000,
                          ylim = C_gs_band,
                          legend = labels,
                          linestyle = method_linestyles,
                          linewidth = [2.0],
                          bintextpos = [0.15, 0.2],
                          bintextsize = 20,
                          figsize = (21, 13),
                          legendfontsize = 20,
                          xaxisticklabelsize = 20,
                          yaxisticklabelsize = 15,
                          yaxislabelsize = 17)

**$\gamma_t(\theta)$ with feedback.** The same grid in real space,
$\gamma_t^{\rm feedback}/\gamma_t^{\rm none}-1$ at the 26 angular bins of the data
vector, from 2.5 to 995.27 arcmin. Each row of panels (one source bin) gets its
own band, fitted to its curves as for $C_\ell^{gs}$. Expect the suppression to
concentrate at small angles, where $\gamma_t$ probes the matter distribution close
to the lens halos.

In [ ]:
# Each entry is the (theta, gammat) pair that plot_gammat_tomo_limber expects
gammat_list = []
for label in labels:
    gammat_list.append((results[label]["theta"], results[label]["gammat"]))
gammat_ref = (ref["theta"], ref["gammat"])

# One fitted band per source row: the slice [:, :, j:j+1] keeps source bin
# j as an axis of length 1, the shape fitted_ratio_band reads
gammat_row_bands = []
for j in range(ref["gammat"].shape[2]):
    row_curves = []
    for label in labels:
        row_curves.append(results[label]["gammat"][:, :, j:j+1])
    row_band = fitted_ratio_band(curves = row_curves,
                                 reference = ref["gammat"][:, :, j:j+1])
    gammat_row_bands.append(row_band)

# bintextpos (0.85, 0.2): the lower-right corner of each panel, which the
# curves leave free (they approach zero at large angles)
cnu.plot_gammat_tomo_limber(theta_gammat = gammat_list,
                            gammat_ref = gammat_ref,
                            param = param,
                            colorbarlabel = "bfmt method",
                            ylim = gammat_row_bands,
                            thetashow = [theta_min_arcmin, theta_max_arcmin],
                            legend = labels,
                            linestyle = method_linestyles,
                            linewidth = [2.0],
                            bintextpos = [0.85, 0.2],
                            bintextsize = 20,
                            figsize = (21, 13),
                            legendfontsize = 20,
                            xaxisticklabelsize = 20,
                            yaxisticklabelsize = 15,
                            yaxislabelsize = 17)